# Library and settings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.chdir("..")
assert os.getcwd().endswith("Code")

In [ ]:
from src.utility import *
from src.utility import CLASS_CATEGORY_RANK_MAP
import src.sparqlgenerator as g

from collections
from collections.abc import Callable
from copy import deepcopy
import csv
import datasets
import exrex
from itertools import product
import numpy as np
import pandas as pd
import pickle
from queue import Queue, Empty
import shutil
from threading import Thread
import time
from tqdm import tqdm
from typing import Union, Literal, Iterable
from warnings import warn, simplefilter
import wikidataintegrator as wdi
import winsound

c:\Users\tharg\anaconda3\envs\gnn_cf\Lib\site-packages\wikidataintegrator\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Suppress some irrelevant FutureWarnings from Pandas
simplefilter("ignore", category=FutureWarning)

## Global variables

In [5]:
__DEBUG_MODE = False

# # Run maximal tasks such as extracting date ranges
RUN_ALL = False
WIKIDATA_USER_AGENT = "Scraping for thesis about source reliability prediction at the University of Copenhagen. qnv757@alumni.ku.dk."

# # Filenames
SOURCE_ID_FILENAME = "source_id.csv"
SOURCE_ID_LOG_FILENAME = "_source_id_log.txt"
NEIGHBOR_DATASETS_DIR = "neighbor_datasets"

### Load data

In [6]:
nl_sources_dataset = datasets.Dataset.load_from_disk(
    os.path.join(DATA_DIR, "nl_sources_dataset")
)
domain_dataset = datasets.Dataset.load_from_disk(
    os.path.join(DATA_DIR, "domain_dataset")
)

## Extracting item IDs from WikiData

### Query WikiData for results similar to NL sources

In [ ]:
# We're going te be limited by 1) serial querying of the API to not
# burden the server and 2) I/O, which releases the GIL, so we'll use a
# single writer thread
__shutdown_signal = None


def __source_itemid_writer(filename, queue, overwrite: bool):
    mode = "a+" if not overwrite else "w"
    with open(filename, mode) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "id",
                "source",
                "sourceLabel",
                "sourceDescription",
                "url",
                *nl_sources_dataset.features,
            ],
        )
        if __empty_file := f.tell() == 0:
            writer.writeheader()
        while True:
            row = queue.get()
            if row is __shutdown_signal:
                queue.task_done()
                break
            else:
                # Rename to fit canonical format.
                row["source"] = row["url"].rpartition("/")[2]
                row["sourceDescription"] = row.pop("description")
                writer.writerow(row)
                queue.task_done()


def __source_itemid_log_writer(queue, num_sources, filename="_source_id_log.txt"):
    """
    Simple log-writer. Doesn't really handle crashes.
    """
    with open(filename, "a+") as f:
        f.write(
            f"Source-ID writing run commenced on {num_sources} sources at {return_timestring()}.\n"
        )
        f.flush()
        while True:
            try:
                time.sleep(15)
                if queue.get() is __shutdown_signal:
                    queue.task_done()
                    termination_string = f"Source-ID writing run successfully ended at {return_timestring()}.\n\n"
                    f.write(termination_string)
                    break
                else:
                    queue.task_done()
                    heartbeat_string = f"Heartbeat at {return_timestring()}.\n"
                    f.write(heartbeat_string)
                    f.flush()
            except Exception:
                print("Logger crashed!")
                f.write(f"Logger encountered unhandled exception!\n")


def __put_query_results_to_queue(row, queue):
    try:
        res_list = wikidata_search(row["source_name"], max_results=1)
        if len(res_list):
            top_res = res_list[0]
            top_res["sourceLabel"] = top_res.pop("label")
            queue.put({**top_res, **row})
    except KeyError:
        return


def write_nl_source_itemid_dataset(
    source_dataset,
    filename=SOURCE_ID_FILENAME,
    logfilename=SOURCE_ID_LOG_FILENAME,
    overwrite=False,
):
    queue = Queue()
    writer_thread = Thread(
        target=__source_itemid_writer, args=[filename, queue, overwrite]
    )
    writer_thread.start()
    logger_thread = Thread(
        target=__source_itemid_log_writer,
        args=[queue, len(source_dataset), logfilename],
    )
    logger_thread.start()
    source_dataset.map(
        lambda row: __put_query_results_to_queue(row, queue),
        desc="Writing NL source ID dataset...",
    )
    queue.put(__shutdown_signal)
    queue.put(__shutdown_signal)
    writer_thread.join()
    logger_thread.join()
    queue.join()
    print("Writing run successfully finished!")


if RUN_ALL:
    write_nl_source_itemid_dataset(nl_sources_dataset, overwrite=True)

In [ ]:
# Let's investigate the outcome of the simple queries in order to
# identify possible general failure modes that we can use to identify
# false hits.
nl_sources_ids_first_run = datasets.load_dataset(
    "csv", data_files=SOURCE_ID_FILENAME, split="train"
)
nl_sources_ids_first_run_df = nl_sources_ids_first_run.to_pandas()
nl_sources_ids_first_run_df

,id,source,sourceLabel,sourceDescription,url,source_name,dataset_label_map,id.1,label,source_dataset,bias,__index_level_0__
0,4,Q161736,comma,punctuation mark,//www.wikidata.org/wiki/Q161736,",",{'ReCoVery': ['0']},4,0,ReCoVery,Extreme left,4
1,13,Q85732306,2019 State of the Nation Address,state of the Nation Address in the Philippines,//www.wikidata.org/wiki/Q85732306,2019 state of the nation address,"{'AVeriTeC': ['Supported', 'Supported', 'Suppo...",13,Supported,AVeriTeC,None,13
2,14,Q65090703,2020 Democratic National Convention,"U.S. political event held in Milwaukee, Wiscon...",//www.wikidata.org/wiki/Q65090703,2020 democratic national convention,{'AVeriTeC': ['Conflicting Evidence/Cherrypick...,14,Conflicting Evidence/Cherrypicking,AVeriTeC,None,14
3,15,Q85737496,2020 elections in India,elections in India in 2020,//www.wikidata.org/wiki/Q85737496,2020 elections,{'AVeriTeC': ['Supported']},15,Supported,AVeriTeC,None,15
4,24,Q238330,4chan,English-language imageboard website,//www.wikidata.org/wiki/Q238330,4chan,"{'AVeriTeC': ['Refuted', 'Refuted']}",24,Refuted,AVeriTeC,None,24
...,...,...,...,...,...,...,...,...,...,...,...,...
5247,9367,Q15918220,Zhong Xing,Ming dynasty politician,//www.wikidata.org/wiki/Q15918220,zhong xin,{'AVeriTeC': ['Refuted']},9367,Refuted,AVeriTeC,None,9367
5248,9370,Q218217,Zoe Lofgren,American politician,//www.wikidata.org/wiki/Q218217,zoe lofgren,"{'Liar-PLUS': ['mostly-true'], 'PolitiFact Fac...",9370,mostly-true,PolitiFact Fact Check,None,9370
5249,9371,Q115267311,Zoë Schiffer,journalist,//www.wikidata.org/wiki/Q115267311,zoe schiffer,"{'ReCoVery': ['1', '1', '1']}",9371,1,ReCoVery,Left-center,9371
5250,9375,Q243833,Zweli Mkhize,South African doctor and politician,//www.wikidata.org/wiki/Q243833,zweli mkhize,"{'AVeriTeC': ['Supported', 'Refuted', 'Refuted...",9375,Supported,AVeriTeC,None,9375


- We first note that we've made some 5000 hits out of 9000 queries, i.e. just above half.
- Failures:
    - Some names have been resolved to e.g. sportspeople or artists, esp. college basketball players, which are likely but not guaranteed to be false hits. Possibel solution: Discard artists, athletes. Note that this may entail discarding celebrities who can sway public opinion, except if we can filter for celebrities.
    - Some sources are generic like "polls" which mapped to "opinion polls".
    - Some names have hit Wikimedia disambiguation pages!
    - Some singular names have mapped to name-entities.
    - Some hits do not have descriptions
    - Some hits are historical people. Possible solutions: Append property "is alive" or a timestamp from after the advent of the 20th century.
    - Some comma-separated designated sources end up with hits such as podcast episodes
    - Similarly, some overly generic sources hit far off field. For example, "town hall audience member" hits with a Key & Peele sketch. Other such entries hit with names of scientific articles.
    - A few patents
    - A couple of named works of art or entertainment e.g. video games.
    - A couple of named artifacts, e.g. ships
    - A couple of named events, e.g. boxing matches
    - Sometimes, abbreviations do seem to throw the query off, e.g. "the bjp" was mapped to an article in which the acronym "BJP" appears.
    - Sometimes, source references and source dating seem to have been mixed up, so some hits are on days of the week or Month
    - A few species of plant
- Somewhat below half of hits seem to be misses, i.e. we may have some 2500/9000 true hits.
- Thoughts: We should probably pick an expansive set of accepted labels rather than a set of exclusions.
- Based on the empirical obvervations, a set of accepted labels could be:
    - Organizations and information entities:
        - Websites
        - NGO's and non-profits
        - Lobbying orgs and lobbyists
        - ThinkTanks
        - Fact-checking organizations
        - Unions and their organizers
        - Companies and brands
        - Activists
    - Political and civil roles:
        - Politicians and state leaders
        - Political committees and task forces
        - Civil servants and functionaries
        - Jurists and judges
    - Educated people and education institutions:
        - Researchers and scientists
        - People with diplomas, e.g. PhD's
        - Universities and educational institutions
        - Economists
    - Conversely, people marked as alternative/fringe science promoters
    - Broader media entities:
        - YouTube channels
        - Podcasts
        - Named social media profiles
        - "Podcast guests" or other loose associations
    - Information centers
        - Museums
        - Databases
        - Archives
    - Journalistic entities:
        - Journalists
        - Analysts
        - Journalistic outlets, incl. television stations and -programs
    - The medical sector:
        - Hospitals
        - Doctors
    - Wealthy and powerful people (who can influence public opinion)
    - Non-fiction filmmakers  # Unfortunately, I don't think there is a category for this
    - Non-fiction authors
    - Clergy and tribal leaders

In [9]:
ACCEPTED_SOURCE_ITEMS = {
    "activist": "Q15253558",
    "business and professional associations, unions": "Q130370871",  # subclasses: trade unions,
    "business magnate": "Q911554",
    "celebrity": "Q211236",  # subclasses: YouTuber
    # "company": "Q783794",  # subclasses: media company
    "content creator": "Q109459317",
    "economist": "Q188094",
    "functionary": "Q1328323",
    "jurist": "Q185351",
    "knowledge worker": "Q1914017",  # Subclasses: scientist
    "leader": "Q1251441",  # Subclasses: oligarch
    "manager": "Q2462658",  # Subclasses: spiritual leader
    "media professional": "Q15980804",  # Subclasses: media personality, journalist
    "misinformation": "Q13579947",
    "misinformation creator": "Q108866986",
    "non-fiction": "Q213051",  # Subclasses: dataset
    "non-fiction writer": "Q15980158",
    "official": "Q599151",
    "organization": "Q43229",  # Subclasses: non-profit organization, NGO, think tank, lobbying org, public company, committees, councils, political institution, educational organization, university, GLAM (galleries, libraries, archives and museums), broadcaster, medical organization,
    "podcast show": "Q24634210",
    "politician": "Q82955",  # subclasses: president, statesperson
    "pseudoscience": "Q483677",
    "publication": "Q732577",  # Subclasses: website
    "scholar": "Q20826540",
}


# Note that these are the accepted properties linking an input source to an
# accepted item (class). We will use other properties to populate the graph.
ACCEPTED_SOURCE_PROPERTIES = {
    "educated at": "P69",
    "instance of": "P31",
    "occupation": "P106",
    "position held": "P39",
    "subclass of": "P279",
}

Filter out the retrieved NL source ID's that don't fit our pattern of accepted sources.

In [ ]:
__shutdown_signal = None
__termination_signal = -1


def __df_concatenator(queue, id_cols: list[str], name: str, df=None, idx=0):
    if df is None:
        df = pd.DataFrame()
    while True:
        x = queue.get()
        if x is __shutdown_signal:
            queue.put(df)
            queue.task_done()
            break
        elif x is __termination_signal:
            print(
                f"Worker received termination signal. Attempting to save dataframe at index {idx}."
            )
            df.to_pickle(in_data_dir(name + "_ckpt.pkl"))
            print(f"saved DF to {os.path.abspath(in_data_dir(name + "_ckpt.pkl"))}")
            with open(in_data_dir(name + "_ckpt_idx.pkl"), "wb") as f:
                pickle.dump(idx, f)
            print(
                f"saved idx to {os.path.abspath(in_data_dir(name + "_ckpt_idx.pkl"))}"
            )
            queue.task_done()
            print(f"Worker terminating.")
            break
        elif isinstance(x, tuple) and len(x[0]):  # I.e. non-empty DF
            res_df, idx = x
            # Strip the ID strings from the returned URLs
            for col in id_cols:
                res_df[col] = res_df[col].str.rpartition("/")[2]
            df = pd.concat([df, res_df])
            queue.task_done()


def construct_dataset_from_queries(
    dataset,
    query_constructor: Callable,
    name: str,
    batch_size: Union[int, tuple] = 20,
    id_cols=["source"],
    additional_cols=None,
    index_col=None,
    bidirectional=False,
    accepted_items_dict=ACCEPTED_SOURCE_ITEMS,
    accepted_properties_dict=ACCEPTED_SOURCE_PROPERTIES,
    human_properties_dict: Optional[dict[str, str]] = None,
    overwrite=False,
    behavior: Union[Literal["raise"], Literal["skip"]] = "raise",
    **kwargs,
):
    """
    Generate a dataset based on joining a dataset from which queries can be generated with WikiData triples.

    If batch_size is a tuple, its first element is the batch-size on items from the dataset, and its second element is the batch-size on the number of accepted properties to check for (to be used in case of timeouts even for tiny batches).

    Note that batch-sizes well below what would provoke a timeout still often provokes some kind of HTTP denial in the WikiDataIntegrator, hence stick to small sizes.
    """
    assert behavior in ["raise", "skip"]
    if "data" in os.path.split(name):
        warn(
            f"Detected folder 'data' in the supplied name {name} in the dataset constructor. Note that the dataset constructor already saves the dataset in the /data/ folder.",
            UserWarning,
        )
    if isinstance(batch_size, tuple):
        data_batch_size = batch_size[0]
        accepted_properties_batch_size = batch_size[1]
        batch_splitting = True
    else:
        batch_splitting = False
        data_batch_size = batch_size
    if not (overwrite) and os.path.exists(in_data_dir(name + "_ckpt_idx.pkl")):
        with open(in_data_dir(name + "_ckpt_idx.pkl"), "rb") as f:
            start_idx = pickle.load(f)
        print(f"Resuming dataset construction from index {start_idx}.")
        if os.path.exists(ckpt := in_data_dir(name + "_ckpt.pkl")):
            df = pd.read_pickle(ckpt)
        else:
            raise RuntimeError(
                f"Found start index {start_idx} but not corresponding ckpt {ckpt} in neighbors dir!\nData dir: {os.listdir(os.path.join(DATA_DIR, NEIGHBOR_DATASETS_DIR))}"
            )
    else:
        start_idx = 0
        df = None
        print("Performing dataset construction from scratch.")
    idxs = np.arange(start_idx, len(dataset), data_batch_size)
    if batch_splitting:
        properties_idxs = np.arange(
            0, len(accepted_properties_dict), accepted_properties_batch_size
        )
    else:
        properties_idxs = [0]
        accepted_properties_batch_size = len(accepted_properties_dict)
    queue = Queue()
    log_queue = Queue()
    worker = Thread(
        target=__df_concatenator, args=[queue, id_cols, name, df, start_idx]
    )

    logger = Thread(target=__query_constructor_log_writer, args=[log_queue])
    worker.start()
    logger.start()

    try:
        for idx in tqdm(idxs, desc="Constructing dataset from queries..."):
            for property_idx in properties_idxs:
                if query_constructor is g.for_neighbor_humans:
                    assert human_properties_dict is not None
                    absolute_kwargs = {
                        "dataset": dataset[idx : idx + data_batch_size],
                        "accepted_relations_dict": dict(
                            list(accepted_properties_dict.items())[
                                property_idx : property_idx
                                + accepted_properties_batch_size
                            ]
                        ),
                        "accepted_properties_dict": human_properties_dict,
                        "accepted_items_dict": accepted_items_dict,
                        **kwargs,
                    }
                else:
                    absolute_kwargs = {
                        "dataset": dataset[idx : idx + data_batch_size],
                        "accepted_items_dict": accepted_items_dict,
                        "accepted_properties_dict": dict(
                            list(accepted_properties_dict.items())[
                                property_idx : property_idx
                                + accepted_properties_batch_size
                            ]
                        ),
                        **kwargs,
                    }
                subject_query = query_constructor(
                    # *query_args,
                    **absolute_kwargs,
                )
                # if __DEBUG_MODE:
                #     print(subject_query)
                #     raise KeyboardInterrupt
                if behavior == "raise":
                    with QueryFailureWatcher(queue, worker):
                        results_df = execute_sparql_query(subject_query)
                else:
                    try:
                        with QueryFailureWatcher(
                            queue, worker, log_queue, logger, behavior=behavior
                        ):
                            results_df = execute_sparql_query(subject_query)
                    except RuntimeError as e:
                        log_queue.put(str(e))
                        winsound.MessageBeep(type=winsound.MB_ICONEXCLAMATION)

                if bidirectional:
                    results_df["subjectClassLabel"] = pd.Series(None)
                for col in ["subjectClassLabel", "objectClassLabel"]:
                    if col not in results_df.columns:
                        results_df[col] = pd.Series(None)
                queue.put((results_df, idx))
                if bidirectional:
                    object_query = query_constructor(
                        seed_role="object",
                        **absolute_kwargs,
                    )
                    if __DEBUG_MODE:
                        print(object_query)
                        raise KeyboardInterrupt
                    if behavior == "raise":
                        with QueryFailureWatcher(queue, worker):
                            results_df = execute_sparql_query(object_query)
                            for col in ["subjectClassLabel", "objectClassLabel"]:
                                if col not in results_df.columns:
                                    results_df[col] = pd.Series(None)
                            queue.put((results_df, idx))
                    else:
                        try:
                            with QueryFailureWatcher(
                                queue, worker, log_queue, logger, behavior=behavior
                            ):
                                results_df = execute_sparql_query(object_query)
                                for col in ["subjectClassLabel", "objectClassLabel"]:
                                    if col not in results_df.columns:
                                        results_df[col] = pd.Series(None)
                                queue.put((results_df, idx))
                        except RuntimeError as e:
                            log_queue.put(str(e))
                            winsound.MessageBeep(type=winsound.MB_ICONEXCLAMATION)
    except Exception:
        queue.put(__termination_signal)
        log_queue.put(__termination_signal)
        raise
    queue.put(__shutdown_signal)
    log_queue.put(__shutdown_signal)
    worker.join()
    logger.join()
    df = queue.get()
    cols_to_keep = list(df.columns)
    if additional_cols:
        assert index_col
        for col in additional_cols:
            cols_to_keep.append(col)
        df = df.join(dataset.set_index(index_col), index_col, "inner", rsuffix="_")
    # Note that duplicate-dropping serves multiple purposes: It is
    # especially important when using inferred URIs, and it drops
    # duplicates from interrupted construction runs, because the
    # checkpointing is not exact.
    df = df[cols_to_keep].reset_index().drop_duplicates()
    output_dataset = datasets.Dataset.from_pandas(df)
    output_dataset.save_to_disk(os.path.join(DATA_DIR, name))
    queue.task_done()
    return output_dataset


def __query_constructor_log_writer(queue, filename="_query_constructor_log.txt"):
    with open(filename, "a+") as f:
        f.write(f"Query data constructor run commenced at {return_timestring()}.\n")
        f.flush()
        while True:
            try:
                msg = queue.get(timeout=15)
                if msg is __shutdown_signal:
                    queue.task_done()
                    termination_string = f"Query data constructor successfully ended at {return_timestring()}.\n\n"
                    f.write(termination_string)
                    break
                elif msg is __termination_signal:
                    queue.task_done()
                    termination_string = f"Query data constructor was prematurely terminated at {return_timestring()}.\n\n"
                    f.write(termination_string)
                    break
                elif isinstance(msg, str):
                    queue.task_done()
                    f.write(msg + "\n\n")
                    f.flush()
            except Empty:
                heartbeat_string = f"Heartbeat at {return_timestring()}.\n"
                f.write(heartbeat_string)
                f.flush()

In [11]:
nl_sources_ids_first_run_df

,id,source,sourceLabel,sourceDescription,url,source_name,dataset_label_map,id.1,label,source_dataset,bias,__index_level_0__
0,4,Q161736,comma,punctuation mark,//www.wikidata.org/wiki/Q161736,",",{'ReCoVery': ['0']},4,0,ReCoVery,Extreme left,4
1,13,Q85732306,2019 State of the Nation Address,state of the Nation Address in the Philippines,//www.wikidata.org/wiki/Q85732306,2019 state of the nation address,"{'AVeriTeC': ['Supported', 'Supported', 'Suppo...",13,Supported,AVeriTeC,None,13
2,14,Q65090703,2020 Democratic National Convention,"U.S. political event held in Milwaukee, Wiscon...",//www.wikidata.org/wiki/Q65090703,2020 democratic national convention,{'AVeriTeC': ['Conflicting Evidence/Cherrypick...,14,Conflicting Evidence/Cherrypicking,AVeriTeC,None,14
3,15,Q85737496,2020 elections in India,elections in India in 2020,//www.wikidata.org/wiki/Q85737496,2020 elections,{'AVeriTeC': ['Supported']},15,Supported,AVeriTeC,None,15
4,24,Q238330,4chan,English-language imageboard website,//www.wikidata.org/wiki/Q238330,4chan,"{'AVeriTeC': ['Refuted', 'Refuted']}",24,Refuted,AVeriTeC,None,24
...,...,...,...,...,...,...,...,...,...,...,...,...
5247,9367,Q15918220,Zhong Xing,Ming dynasty politician,//www.wikidata.org/wiki/Q15918220,zhong xin,{'AVeriTeC': ['Refuted']},9367,Refuted,AVeriTeC,None,9367
5248,9370,Q218217,Zoe Lofgren,American politician,//www.wikidata.org/wiki/Q218217,zoe lofgren,"{'Liar-PLUS': ['mostly-true'], 'PolitiFact Fac...",9370,mostly-true,PolitiFact Fact Check,None,9370
5249,9371,Q115267311,Zoë Schiffer,journalist,//www.wikidata.org/wiki/Q115267311,zoe schiffer,"{'ReCoVery': ['1', '1', '1']}",9371,1,ReCoVery,Left-center,9371
5250,9375,Q243833,Zweli Mkhize,South African doctor and politician,//www.wikidata.org/wiki/Q243833,zweli mkhize,"{'AVeriTeC': ['Supported', 'Refuted', 'Refuted...",9375,Supported,AVeriTeC,None,9375


In [ ]:
# We will not parallelize for the sake of the servers.
if RUN_ALL:
    accepted_nl_sources = construct_dataset_from_queries(
        nl_sources_ids_first_run_df,
        for_valid_sources,
        "accepted_nl_wikidata_sources",
        batch_size=100,
        accepted_properties_dict=ACCEPTED_SOURCE_PROPERTIES,
        accepted_items_dict=ACCEPTED_SOURCE_ITEMS,
    )
    accepted_nl_sources_df = (
        accepted_nl_sources.to_pandas()
        .merge(
            nl_sources_ids_first_run_df,
            on=["source", "sourceLabel", "sourceDescription"],
            how="inner",
            suffixes=("_accepted", "_all"),
        )[
            [
                "source_name_accepted",
                "source",
                "sourceLabel",
                "sourceDescription",
                "sourceClassLabel",
                "url",
                "bias",
                "dataset_label_map",
            ]
        ]
        .rename(columns={"source_name_accepted": "source_name"})
    )

    accepted_nl_sources = datasets.Dataset.from_pandas(accepted_nl_sources_df)
    accepted_nl_sources.save_to_disk(in_data_dir("accepted_nl_wikidata_sources"))
else:
    accepted_nl_sources = datasets.load_from_disk(
        in_data_dir("accepted_nl_wikidata_sources")
    )
    accepted_nl_sources_df = accepted_nl_sources.to_pandas()

In [13]:
accepted_nl_sources_df

,source_name,source,sourceLabel,sourceDescription,sourceClassLabel,url,bias,dataset_label_map
0,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,organization,//www.wikidata.org/wiki/Q40464,None,"{'AVeriTeC': ['Refuted'], 'MM-COVID19': ['fake..."
1,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,organization,//www.wikidata.org/wiki/Q40464,None,{'AVeriTeC': ['Supported']}
2,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,company,//www.wikidata.org/wiki/Q40464,None,"{'AVeriTeC': ['Refuted'], 'MM-COVID19': ['fake..."
3,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,company,//www.wikidata.org/wiki/Q40464,None,{'AVeriTeC': ['Supported']}
4,abc,Q169889,American Broadcasting Company,American broadcast television network owned by...,organization,//www.wikidata.org/wiki/Q169889,None,"{'AVeriTeC': ['Refuted', 'Supported', 'Support..."
...,...,...,...,...,...,...,...,...
43669,youtube videos,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'PolitiFact Fact Check': ['false', 'false', '..."
43670,youtube video,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'DRUID+': ['False', 'False', 'False', 'False'..."
43671,youtube video,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'PolitiFact Fact Check': ['false', 'false', '..."
43672,youtube videos,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'DRUID+': ['False', 'False', 'False', 'False'..."


### Query WikiData for items with URLs from our dataset as official website.

#### Add categorical columns

In [14]:
accepted_nl_sources_df

,source_name,source,sourceLabel,sourceDescription,sourceClassLabel,url,bias,dataset_label_map
0,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,organization,//www.wikidata.org/wiki/Q40464,None,"{'AVeriTeC': ['Refuted'], 'MM-COVID19': ['fake..."
1,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,organization,//www.wikidata.org/wiki/Q40464,None,{'AVeriTeC': ['Supported']}
2,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,company,//www.wikidata.org/wiki/Q40464,None,"{'AVeriTeC': ['Refuted'], 'MM-COVID19': ['fake..."
3,afp,Q40464,Agence France-Presse,international news agency headquartered in Paris,company,//www.wikidata.org/wiki/Q40464,None,{'AVeriTeC': ['Supported']}
4,abc,Q169889,American Broadcasting Company,American broadcast television network owned by...,organization,//www.wikidata.org/wiki/Q169889,None,"{'AVeriTeC': ['Refuted', 'Supported', 'Support..."
...,...,...,...,...,...,...,...,...
43669,youtube videos,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'PolitiFact Fact Check': ['false', 'false', '..."
43670,youtube video,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'DRUID+': ['False', 'False', 'False', 'False'..."
43671,youtube video,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'PolitiFact Fact Check': ['false', 'false', '..."
43672,youtube videos,Q63412991,YouTube video,"software resource that represents a video, its...",publication,//www.wikidata.org/wiki/Q63412991,None,"{'DRUID+': ['False', 'False', 'False', 'False'..."


In [ ]:
def _add_category_column(row, target="source"):
    """
    Given a dataframe with the target ClassLabel column, add a corresponding column of the highest ranked category that the set of labels belongs to.
    """

    if None not in row[f"{target}ClassLabels"]:
        row[f"{target}Category"] = min(
            (
                CLASS_CATEGORY_RANK_MAP[_try_clean_identifier_label(label)]
                for label in row[f"{target}ClassLabels"]
                if _try_clean_identifier_label(label) is not None
            ),
            key=lambda category_rank_pair: category_rank_pair[1],
        )[0]

    else:

        row[f"{target}Category"] = None

    return row


def _try_clean_identifier_label(label: str) -> str:
    """
    Some labels are badly formatted. In particular, some are identifiers rather than labels. Try to patch that issue here, if possible.
    """

    if label in CLASS_CATEGORY_RANK_MAP.keys():
        return label
    elif label in ACCEPTED_SOURCE_ITEMS.values():
        for k, v in ACCEPTED_SOURCE_ITEMS.items():
            if v == label:
                return k
    else:
        return None


def aggregate_class_labels(df, target="source") -> pd.DataFrame:
    """
    Take an (exploded) dataframe of multiple labels per source and convert it to a dataframe where the labels are aggregated in arrays.
    """
    result = df.groupby(target).first()
    result[f"{target}ClassLabels"] = df.groupby(target)[f"{target}ClassLabel"].unique()
    return result.reset_index()


def aggregate_df_class_labels(df, target="source", **kwargs) -> pd.DataFrame:
    """

    Take an exploded dataframe with multiple labels per source and compress it such that each source is associated with a list of labels, further add a column for the highest ranked category the source belongs to.
    """
    return aggregate_class_labels(df, target=target).apply(
        lambda row: _add_category_column(row, target=target, **kwargs), axis=1
    )


accepted_nl_sources_df = aggregate_df_class_labels(accepted_nl_sources_df)

#### Merge constructed datasets

Being crowdsourced, formatting on WikiData can be inconsistent. The exact URI structures of given websites vary. Hence we will have to try a superset to be thorough.  

Create an exploded DataFrame where each row has a domain mapped to one of a handful of candidate mutations of the domain as URI. This is a big and wasteful DataFrame, but we will only need it until we merge the WikiData results.

#### Expand domain strings to possibly matching common URLs  

In [16]:
common_domain_pattern = "(https?://)?(www\\.)?"


def _generate_list_of_possible_uris(
    domain, pattern=common_domain_pattern, inferred=True
):
    if inferred:
        return [
            "".join(substring_tuple)
            for substring_tuple in product(
                exrex.generate(pattern), [domain, domain + "/"]
            )
        ]

    else:
        return [domain, domain + "/"]


def _add_list_of_uris_column_to_row(row):
    row["list_of_uris_to_try"] = _generate_list_of_possible_uris(row["source_name"])
    return row


domain_dataset = domain_dataset.map(_add_list_of_uris_column_to_row)

exploded_domain_df = (
    domain_dataset.to_pandas()
    .explode("list_of_uris_to_try")
    .rename(
        columns={
            "list_of_uris_to_try": "website",
            "originalDomain": "source_name",
        }
    )
)
exploded_domain_df

,index,source_name,dataset_label_map,id,label,source_dataset,bias,website
0,0,0585news.com,{'Domain Ratings': [0.261046492905716]},10070,0.261046492905716,Domain Ratings,None,0585news.com
0,0,0585news.com,{'Domain Ratings': [0.261046492905716]},10070,0.261046492905716,Domain Ratings,None,0585news.com/
0,0,0585news.com,{'Domain Ratings': [0.261046492905716]},10070,0.261046492905716,Domain Ratings,None,www.0585news.com
0,0,0585news.com,{'Domain Ratings': [0.261046492905716]},10070,0.261046492905716,Domain Ratings,None,www.0585news.com/
0,0,0585news.com,{'Domain Ratings': [0.261046492905716]},10070,0.261046492905716,Domain Ratings,None,http://0585news.com
...,...,...,...,...,...,...,...,...
13306,13306,zvw.de,{'Domain Ratings': [0.696441085914132]},2711,0.696441085914132,Domain Ratings,None,http://www.zvw.de/
13306,13306,zvw.de,{'Domain Ratings': [0.696441085914132]},2711,0.696441085914132,Domain Ratings,None,https://zvw.de
13306,13306,zvw.de,{'Domain Ratings': [0.696441085914132]},2711,0.696441085914132,Domain Ratings,None,https://zvw.de/
13306,13306,zvw.de,{'Domain Ratings': [0.696441085914132]},2711,0.696441085914132,Domain Ratings,None,https://www.zvw.de


In [17]:
if RUN_ALL:
    # Construct a dataset with WikiData hits from the exploded DataFrame
    wikidata_domains_df = construct_dataset_from_queries(
        exploded_domain_df,
        for_owners_of_official_websites,
        "wikidata_sources_from_domains",
        batch_size=10,
        uri_is_inferred=True,
    ).to_pandas()
    # Merge it with the exploded DataFrame to retrieve the dataset_label_maps
    wikidata_domains_df_merged = wikidata_domains_df.merge(
        exploded_domain_df,
        on="website",
        how="inner",
    )
    # Construct another DataFrame with the classes of each source
    wikidata_domains_label_df = construct_dataset_from_queries(
        wikidata_domains_df,
        g.for_class_labels,
        "wikidata_domains_label_df",
        batch_size=50,
        accepted_items_dict=ACCEPTED_SOURCE_ITEMS,
        accepted_properties_dict=ACCEPTED_SOURCE_PROPERTIES,
    ).to_pandas()
    # Aggregate the labels and merge with the previous DataFrame
    wikidata_domains = datasets.Dataset.from_pandas(
        aggregate_df_class_labels(
            wikidata_domains_label_df.merge(
                wikidata_domains_df_merged,
                on="source",
                how="inner",
            )
        )[
            [
                "source",
                "sourceLabel",
                "sourceDescription",
                "sourceClassLabel",
                "sourceCategory",
                "website",
                "bias",
                "dataset_label_map",
            ]
        ]
    )
    wikidata_domains.save_to_disk(
        os.path.join(DATA_DIR, "wikidata_sources_from_domains")
    )
else:
    wikidata_domains = datasets.load_from_disk(
        os.path.join(DATA_DIR, "wikidata_sources_from_domains")
    )

wikidata_domains.to_pandas()

,source,sourceLabel,sourceDescription,sourceClassLabel,sourceCategory,website,bias,dataset_label_map
0,Q100281204,The Rock Island Argus,"newspaper published in Rock Island, Illinois",publication,organization,https://qconline.com/,None,{'Domain Ratings': [0.581255507206459]}
1,Q100285264,The Town Talk,"newspaper published in Alexandria, Louisiana",organization,organization,https://www.thetowntalk.com,None,{'Domain Ratings': [0.879334607656032]}
2,Q100289468,The Coastland Times,"newspaper published in Manteo, Dare County, No...",publication,organization,https://www.thecoastlandtimes.com,None,{'Domain Ratings': [0.653315560209103]}
3,Q100298819,The Herald-Palladium,"newspaper published in Saint Joseph, Michigan",publication,organization,http://www.heraldpalladium.com/,None,{'Domain Ratings': [0.57078011158889]}
4,Q100320635,American Stroke Association,division of the American Heart Association foc...,organization,organization,https://www.stroke.org/,None,{'Domain Ratings': [0.676772122938108]}
...,...,...,...,...,...,...,...,...
5651,Q99701325,Irish Daily Mirror,Irish daily newspaper,organization,organization,https://www.irishmirror.ie/,None,"{'GossipCop': [False, False, False, False, Fal..."
5652,Q99766775,History Extra,website,publication,data,https://www.historyextra.com,None,{'GossipCop': [True]}
5653,Q99824437,Redorbit,web site,publication,data,http://www.redorbit.com/,None,{'Domain Ratings': [0.284754150463462]}
5654,Q998264,"Tradition, Family, Property",international movement of organizations of Tra...,organization,organization,https://www.tfp.org/,extreme-right,"{'Domain Ratings': [0.235355269400232], 'News ..."


Now query WikiData for items that have these domains as official website. Note that the speed of this process would be an order of magnitude faster if we stuck to original URLs and didn't try out all these candidate URIs.

In [18]:
mergeable_nl_sources = (
    accepted_nl_sources_df.groupby("source")
    .first()
    .reset_index()
    .rename(
        columns={
            "sourceLabel": "sourceLabel",
            "sourceDescription": "sourceDescription",
        }
    )
    .set_index(["source", "sourceLabel", "sourceDescription", "sourceCategory"])
)

mergeable_domain_sources = (
    wikidata_domains.to_pandas()
    .groupby("source")
    .first()
    .reset_index()
    .rename(
        columns={
            "sourceLabel": "sourceLabel",
            "sourceDescription": "sourceDescription",
        }
    )
    .set_index(["source", "sourceLabel", "sourceDescription", "sourceCategory"])
)

wikifake_df = mergeable_nl_sources.merge(
    mergeable_domain_sources,
    on=["source", "sourceLabel", "sourceDescription", "sourceCategory"],
    how="outer",
    suffixes=("_from_nl", "_from_domain"),
).reset_index()


wikifake_df["bias_from_nl"] = wikifake_df["bias_from_nl"].str.lower()
wikifake_df["bias_from_nl"].str.replace("extreme-right", "extreme right")
wikifake_df["bias_from_nl"].str.replace("extreme-left", "extreme left")
wikifake_df["bias_from_domain"] = wikifake_df["bias_from_domain"].str.lower()
wikifake_df["bias_from_domain"].str.replace("extreme-right", "extreme right")
wikifake_df["bias_from_domain"].str.replace("extreme-left", "extreme left")
wikifake_df

,source,sourceLabel,sourceDescription,sourceCategory,source_name,sourceClassLabel_from_nl,url,bias_from_nl,dataset_label_map_from_nl,sourceClassLabels,sourceClassLabel_from_domain,website,bias_from_domain,dataset_label_map_from_domain
0,Q1000479,Boston Children's Hospital,"children's hospital in Boston, Massachusetts, ...",organization,boston children's hospital,organization,//www.wikidata.org/wiki/Q1000479,None,{'FakeHealth': ['4']},[organization],NaN,NaN,NaN,NaN
1,Q100233612,Nina Perales,"Vice President of Litigation, Mexican American...",person,nina perales,jurist,//www.wikidata.org/wiki/Q100233612,None,"{'Liar-PLUS': ['mostly-true'], 'PolitiFact Fac...",[jurist],NaN,NaN,NaN,NaN
2,Q100250975,Charles Bethea,NaN,person,charles bethea,media professional,//www.wikidata.org/wiki/Q100250975,left,{'ReCoVery': ['1']},[media professional],NaN,NaN,NaN,NaN
3,Q100281204,The Rock Island Argus,"newspaper published in Rock Island, Illinois",organization,NaN,NaN,NaN,NaN,NaN,NaN,publication,https://qconline.com/,None,{'Domain Ratings': [0.581255507206459]}
4,Q100285264,The Town Talk,"newspaper published in Alexandria, Louisiana",organization,NaN,NaN,NaN,NaN,NaN,NaN,organization,https://www.thetowntalk.com,None,{'Domain Ratings': [0.879334607656032]}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9269,Q99774723,Cindy O'Laughlin,American politician,person,cindy o’laughlin,politician,//www.wikidata.org/wiki/Q99774723,None,"{'PolitiFact Fact Check': ['false', 'half-true']}","[politician, manager]",NaN,NaN,NaN,NaN
9270,Q99824437,Redorbit,web site,data,NaN,NaN,NaN,NaN,NaN,NaN,publication,http://www.redorbit.com/,None,{'Domain Ratings': [0.284754150463462]}
9271,Q998264,"Tradition, Family, Property",international movement of organizations of Tra...,organization,NaN,NaN,NaN,NaN,NaN,NaN,organization,https://www.tfp.org/,extreme-right,"{'Domain Ratings': [0.235355269400232], 'News ..."
9272,Q99841649,TheaterMania,website,data,NaN,NaN,NaN,NaN,NaN,NaN,publication,https://www.theatermania.com/,None,{'GossipCop': [True]}


In [19]:
def __merge_label_map_dicts(d1, d2):
    # Also expand the dictionary to contain None-values for missing
    # datasets
    merged = {}
    for key in list(d1.keys()) + list(d2.keys()):
        vals1 = d1.get(key, []) if d1.get(key) is not None else []
        vals2 = d2.get(key, []) if d2.get(key) is not None else []
        merged.update({key: vals1 + vals2 if len(vals1 + vals2) else None})
    return merged


def _merge_row_label_maps(row):
    nans = row.isna()
    merged = dict.fromkeys(ALL_DATASETS)
    if nans["dataset_label_map_from_nl"]:
        merged.update(eval(row["dataset_label_map_from_domain"]))
        return merged
    elif not (nans["dataset_label_map_from_domain"]):
        from_nl = row["dataset_label_map_from_nl"]
        from_domain = row["dataset_label_map_from_domain"]
        dict_from_nl = eval(from_nl)
        dict_from_domain = eval(from_domain)
        merged.update(__merge_label_map_dicts(dict_from_nl, dict_from_domain))
        return merged
    else:
        merged.update(eval(row["dataset_label_map_from_nl"]))
        return merged


def _merge_row_class_labels(row):
    nans = row.isna()
    if nans["sourceClassLabel_from_nl"]:
        return row["sourceClassLabel_from_domain"]
    elif not (nans["sourceClassLabel_from_domain"]):
        from_nl = set(row["sourceClassLabel_from_nl"])
        from_domain = set(row["sourceClassLabel_from_domain"])
        return np.array(set.union(from_nl, from_domain))
    else:
        return row["sourceClassLabel_from_nl"]


def _merge_row_bias_labels(row):
    nans = row.isna()
    if nans["bias_from_nl"]:
        if nans["bias_from_domain"]:
            return None
        else:
            return [row["bias_from_domain"]]
    elif nans["bias_from_domain"]:
        return [row["bias_from_nl"]]
    elif row["bias_from_nl"] != row["bias_from_domain"]:
        return [row["bias_from_nl"], row["bias_from_domain"]]
    else:
        return [row["bias_from_nl"]]


def merge_domain_nl_columns(dataset):
    out = dataset
    # You need the string conversion if you want to convert this to a HF
    # dataset
    out["merged_dataset_label_map"] = out.apply(_merge_row_label_maps, axis=1).astype(
        "string"
    )
    out["sourceClassLabels"] = out.apply(_merge_row_class_labels, axis=1).astype(
        "string"
    )
    out["merged_bias"] = out.apply(_merge_row_bias_labels, axis=1).astype("string")
    out = out.drop(columns=["sourceClassLabel_from_nl", "sourceClassLabel_from_domain"])
    return out


merged_wikifake_df = merge_domain_nl_columns(wikifake_df)
merged_wikifake_df

,source,sourceLabel,sourceDescription,sourceCategory,source_name,url,bias_from_nl,dataset_label_map_from_nl,sourceClassLabels,website,bias_from_domain,dataset_label_map_from_domain,merged_dataset_label_map,merged_bias
0,Q1000479,Boston Children's Hospital,"children's hospital in Boston, Massachusetts, ...",organization,boston children's hospital,//www.wikidata.org/wiki/Q1000479,None,{'FakeHealth': ['4']},organization,NaN,NaN,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>
1,Q100233612,Nina Perales,"Vice President of Litigation, Mexican American...",person,nina perales,//www.wikidata.org/wiki/Q100233612,None,"{'Liar-PLUS': ['mostly-true'], 'PolitiFact Fac...",jurist,NaN,NaN,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>
2,Q100250975,Charles Bethea,NaN,person,charles bethea,//www.wikidata.org/wiki/Q100250975,left,{'ReCoVery': ['1']},media professional,NaN,NaN,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",['left']
3,Q100281204,The Rock Island Argus,"newspaper published in Rock Island, Illinois",organization,NaN,NaN,NaN,NaN,publication,https://qconline.com/,None,{'Domain Ratings': [0.581255507206459]},"{'AVeriTeC': None, 'Domain Ratings': [0.581255...",<NA>
4,Q100285264,The Town Talk,"newspaper published in Alexandria, Louisiana",organization,NaN,NaN,NaN,NaN,organization,https://www.thetowntalk.com,None,{'Domain Ratings': [0.879334607656032]},"{'AVeriTeC': None, 'Domain Ratings': [0.879334...",<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9269,Q99774723,Cindy O'Laughlin,American politician,person,cindy o’laughlin,//www.wikidata.org/wiki/Q99774723,None,"{'PolitiFact Fact Check': ['false', 'half-true']}",politician,NaN,NaN,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>
9270,Q99824437,Redorbit,web site,data,NaN,NaN,NaN,NaN,publication,http://www.redorbit.com/,None,{'Domain Ratings': [0.284754150463462]},"{'AVeriTeC': None, 'Domain Ratings': [0.284754...",<NA>
9271,Q998264,"Tradition, Family, Property",international movement of organizations of Tra...,organization,NaN,NaN,NaN,NaN,organization,https://www.tfp.org/,extreme-right,"{'Domain Ratings': [0.235355269400232], 'News ...","{'AVeriTeC': None, 'Domain Ratings': [0.235355...",['extreme-right']
9272,Q99841649,TheaterMania,website,data,NaN,NaN,NaN,NaN,publication,https://www.theatermania.com/,None,{'GossipCop': [True]},"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>


In [20]:
wikifake = datasets.Dataset.from_pandas(merged_wikifake_df)
wikifake_mini_df = wikifake_df[
    [
        "source",
        "sourceLabel",
        "sourceDescription",
        "sourceCategory",
        "website",
        "merged_dataset_label_map",
        "merged_bias",
    ]
]
if RUN_ALL:
    wikifake.save_to_disk(os.path.join(DATA_DIR, "wikifake"))
    wikifake_df.to_csv("wikifake.csv")
    wikifake_df[~wikifake_df["sourceLabel"].isin(CLASS_TYPE_SOURCES_TO_DROP)]
    datasets.Dataset.from_pandas(wikifake_mini_df).save_to_disk(
        in_data_dir("wikifake_mini")
    )
    wikifake_mini_df.to_csv("wikifake_mini.csv")
wikifake_mini_df

,source,sourceLabel,sourceDescription,sourceCategory,website,merged_dataset_label_map,merged_bias
0,Q1000479,Boston Children's Hospital,"children's hospital in Boston, Massachusetts, ...",organization,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>
1,Q100233612,Nina Perales,"Vice President of Litigation, Mexican American...",person,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>
2,Q100250975,Charles Bethea,NaN,person,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",['left']
3,Q100281204,The Rock Island Argus,"newspaper published in Rock Island, Illinois",organization,https://qconline.com/,"{'AVeriTeC': None, 'Domain Ratings': [0.581255...",<NA>
4,Q100285264,The Town Talk,"newspaper published in Alexandria, Louisiana",organization,https://www.thetowntalk.com,"{'AVeriTeC': None, 'Domain Ratings': [0.879334...",<NA>
...,...,...,...,...,...,...,...
9269,Q99774723,Cindy O'Laughlin,American politician,person,NaN,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>
9270,Q99824437,Redorbit,web site,data,http://www.redorbit.com/,"{'AVeriTeC': None, 'Domain Ratings': [0.284754...",<NA>
9271,Q998264,"Tradition, Family, Property",international movement of organizations of Tra...,organization,https://www.tfp.org/,"{'AVeriTeC': None, 'Domain Ratings': [0.235355...",['extreme-right']
9272,Q99841649,TheaterMania,website,data,https://www.theatermania.com/,"{'AVeriTeC': None, 'Domain Ratings': None, 'DR...",<NA>


## Non-seed graph population

High level huge set of possibly useful WikiData item and property IDs.

In [21]:
# Sure, we could automate this process using ingenuity and the multithreaded
# search above, but that would hammer the Wikidata servers.
items_directly_related_to_source_reliability = {
    "belief": "Q34394",
    "certainty": "Q1520777",
    "claim": "Q95972591",
    "confidence": "Q4116378",
    "conviction": "Q1048607",
    "correctness": "Q1648040",
    "credibility": "Q1530061",
    "distrust": "Q621922",
    "doubt": "Q34302",
    "fairness": "Q771773",
    "has goal": "P3712",
    "impartiality": "Q3149193",
    "Media Bias/Fact Check": "Q60741379",
    "neutral point of view": "Q2218846",
    # As of a news medium; the philosophical concept is distinct.
    "objectivity": "Q3214913",
    "opinion": "Q3962655",
    "point of view": "Q12558574",
    "soundness": "Q693083",
    "source known to be reliable": "Q118330126",
    "source known to be unreliable": "Q22979588",
    "source of information": "Q3523102",
    # NOTE: For Wikimedia in particular
    "source reliability assessment": "Q118286383",
    "subjectivity": "Q1139554",
    "suspicion": "Q516519",
    "trust": "Q659974",
    "unfairness": "Q47002783",
    "unbelief": "Q16547842",
    "uncertainty": "Q13649246",
    "world view": "Q49447",  # Said to be the same as-
    "worldview": "Q2030462",
    "wrongness": "Q115430545",
}

items_for_appraisal = {
    "appraisal": "Q1663833",
    "assessment": "Q851176",
    "certification": "Q374814",
    "certificate": "Q196756",
    "certified certificate": "Q12219184",
    "confers": "Q101072499",  # award or certification that is conferred by this organization
    "endorsement": "Q265871",
    "evaluation": "Q1379672",
    "lie detection": "Q60920",
    "peer review": "Q215028",
    "reliability": "Q15139812",
    "review": "Q265158",
    "test": "Q1003030",
    "test result": "Q91615379",
    "testability": "Q2149495",
    "validation": "Q106425867",
    "validity": "Q18757",
    "verification": "Q953429",
}

properties_for_appraisal = {
    "assessment": "P5021",
    "complies with": "P5009",
    "endorsed by": "P8001",
    "Media Bias/Fact Check ID": "P9852",
    "review of": "P6977",
    "reviewed by": "P4032",
    "social media followers": "P8687",
    "statement disputed by": "P1310",
    "statement supported by": "P3680",
}

# Contains official high-level labels for types of work, such as
# misinformation vs information and science vs fringe!
items_for_structural_vettedness = {
    "academia": "Q1211427",
    # Inverse of "academic degree"
    "academic degree held by": "Q66499315",
    "academic editor": "Q104629540",  # Canonically: editor
    "academic journal": "Q737498",
    "academic journal article": "Q18918145",
    "academic work": "Q10383930",
    "academic writing": "Q4119870",
    "academic professional": "Q66666685",
    "academic publishing": "Q5246046",
    "academician": "Q414528",
    "researcher": "Q1650915",
    "scholarly article": "Q13442814",
    "scholarly method": "Q17079481",
    "scholarly peer review": "Q7096397",
    # Superclass of academic work that allows non-academic work
    "scholarly work": "Q55915575",
    "science communication": "Q472328",
    "science communicator": "Q15143191",
    "scientific evidence": "Q3320364",
    "scientific journal": "Q5633421",
    "scientific literature": "Q12042160",
    "scientific method": "Q46857",
    "scientific publication": "Q591041",
    "scientific research": "Q2673285",
    "scientific skepticism": "Q929771",
    "scientific style": "Q1965486",
    "scientist": "Q901",
    "science writer": "Q3745071",
    "scientific writing": "Q11818468",
    "work of science": "Q11826511",
    "alternative facts": "Q28478874",
    "alternative media": "Q192855",
    "anecdotal evidence": "Q143636",
    "consensus": "Q202722",
    "conspiracy": "Q930164",
    "conspiracy theorist": "Q19831149",
    "conspiracy theory": "Q159535",
    "debunker": "Q3704426",
    "disinformation": "Q189656",
    "disinformation attack": "Q104858074",
    "Disinformation research": "Q133816136",
    "echo chamber": "Q3334446",
    "elite": "Q192797",
    "fact-checker": "Q920093",
    "fact-checking": "Q59555084",
    "fake news": "Q28549308",
    "fringe science": "Q1140308",
    "fringe theory": "Q5504601",
    "heterodoxy": "Q1992247",
    "mainstream": "Q18957",
    "mainstream media": "Q5292797",
    "malinformation": "Q111175123",  # True but misleading information
    "manipulation": "Q106466760",
    "misinformation": "Q13579947",
    "misinformation correction": "Q105047006",
    # information declared officially by a recognized person or
    # authority (value used with qualifier P5102 and P7452)
    "official": "Q29509043",
    "orthodoxy": "Q383258",
    "propaganda": "Q7281",
    "propaganda campaign": "Q131827787",
    "propagandist": "Q1685363",
    "pseudo-scholarship": "Q7254446",
    "pseudoscience": "Q483677",
    "pseudoscientist": "Q22082749",
    "right-wing alternative media": "Q4736527",
    "rumor": "Q878352",
    # information which is assumed to be well-known, but not declared
    # officially by a recognized person or authority (value used with
    # qualifier P5102 and P2241)
    "unofficial": "Q29509080",
}

items_for_citation = {
    "bibliographic citation": "Q59156842",
    "bibliographic record": "Q1774089",
    "bibliographic reference": "Q10358455",
    "bibliography": "Q1631107",
    "citation": "Q1713",
    "citation practices": "Q113997140",
    "citation rate": "Q65807469",
    "cited by": "Q65969964",
    # As a subclass of primary research
    "original research": "Q1813536",
    "primary research": "Q6505834",
    "referenced as source by": "Q67955220",
    "quotation": "Q206287",
    "quoted by": "Q66204407",  # inverse of "quotes"
    "reference": "Q121769",
    "secondary research": "Q1629267",
    "uncitedness": "Q2492572",
}

properties_for_citation = {
    "inferred from": "P3452",
    # "value suitable for "stated in" (P248) when property is used in
    # references to refer to the primary publication/website/database
    # associated with the property"
    # "applicable 'stated in' value": "P9073",
    "cites work": "P2860",
    "described by source": "P1343",
    "directs readers to": "P13269",
    "quotation": "P1683",  # I.e. of a source
    "quotation or excerpt": "P7081",
    "quotes work": "P6166",
    "reference URL": "P854",
    "reply to": "P2675",
    # "stated in": "P248",
    # That is, when a text asserts that another text asserts that a
    # third text says  something; to be used with the qualifier P248
    "stated in source according to": "P12132",
    "type of reference": "P3865",
}

items_diretly_related_to_truthfulness = {
    "critical thinking": "Q843894",
    "deception": "Q170028",
    "disagreement": "Q104813442",
    "empirical evidence": "Q83021",
    "evidence": "Q1347572",
    "fact": "Q188572",
    "false": "Q5432619",  # As a truth value
    "false statement": "Q17949",
    "falsifiability": "Q220888",
    "hidden agenda": "Q121409158",
    "honesty": "Q337349",
    "liar": "Q43670229",
    "lie": "Q4925193",
    "objection": "Q2090618",
    "quality": "Q185957",
    "replicability": "Q63285799",
    # I.e. something like a subclass of replicability, where the
    # material details can be replicated
    "reproducibility": "Q1425625",
    "true": "Q16751793",  # As a truth value
    "truth": "Q7949",
}

items_related_to_ideology = {
    "has this political ideology": "Q99970346",
    "ideology": "Q7257",
    "member of political party": "Q11388835",
    "narrative": "Q25393572",
    "non-politics": "Q7048979",
    "party platform": "Q1501855",
    "party politics": "Q1647683",
    "personal narrative": "Q17108519",
    "political activist": "Q11499147",
    "political agenda": "Q2920329",
    "political alignment": "Q28819924",  # I.e. on the spectrum
    "political ideology": "Q12909644",
    "political opposition": "Q192852",
    "political organization": "Q7210356",
    # May be a duplicate of the identically named entry?
    "political organization (2)": "Q10341436",
    "political party": "Q7278",
    "political polarization": "Q3082509",
    "political position": "Q20045456",
    "political spectrum": "Q210918",
    "politician": "Q82955",
    "politics": "Q7163",
    "view": "Q109361341",
}

properties_for_ideology = {
    "member of political party": "P102",
    "political alignment": "P1387",
    "political ideology": "P1142",
}

items_for_personnel = {
    "administrator": "Q16532929",
    "board": "Q9387106",
    "board of directors": "Q188628",
    "board of directors member": "Q55168644",
    "board of trustees": "Q96597574",
    "board of trustees (2)": "Q131747868",
    "board member": "Q2824523",
    "CEO of": "Q76451917",  # Inverse of CEO
    "chairman of the executive board": "Q2534525",
    "chairperson": "Q140686",
    "chief executive officer": "Q484876",
    "chief manager": "Q20920693",
    "co-president": "Q105490190",
    "director": "Q1162163",
    "director (2)": "Q11573524",
    "editor": "Q1607826",
    "editor of": "Q66004589",
    "executive board": "Q897395",
    "executive director": "Q267936",
    "founder of": "Q65972149",
    "founding chairperson": "Q132794751",
    "head of state": "Q48352",
    "leader of organization": "Q25713832",
    "maintainer of": "Q104708313",  # Inverse of maintains
    "manager": "Q2462658",
    "member of the board of trustees": "Q113133048",
    "non-executive director": "Q837703",
    # social role with a set of powers and responsibilities within an
    # organization
    "position": "Q4164871",
    "position held by": "Q65970006",  # Inverse of holds position
    "president": "Q1255921",  # of an org
    "role": "Q214339",
    "served as director / manager of": "Q97494654",
}

properties_for_personnel = {
    "chairperson": "P488",
    "director / manager": "P1037",
    "position holder": "P1308",
    "board member": "P3320",
    "chief executive officer": "P169",
    "chief operating officer": "P1789",
    "editor": "P98",
    "editor-in-chief": "P5769",
    "employer": "P108",
    "general secretary": "P3975",
    "head of government": "P6",
    "member of": "P463",
    # some or all members of this group share the occupation
    "members have occupation": "P3989",
    "owned by": "P127",
    "owner of": "P1830",
    "position held": "P39",
    "position held by head of the organization": "P2388",
}

items_for_power_relations = {
    "benefactor": "Q4887411",
    "beneficiary": "Q2596417",
    "donation": "Q1124860",
    "donor": "Q1968442",
    "division": "Q334453",  # Of company
    "foundation": "Q157031",  # Type of org
    "front organization": "Q1470315",
    "fundraising": "Q1161355",
    "gift": "Q184303",
    "governance": "Q1553864",
    "governing body": "Q895526",
    "government": "Q7188",
    "government administration": "Q3754526",
    "government agency": "Q327333",
    "granted by": "Q122497940",  # Inverse of
    "has commissioned": "Q66004579",  # Inverse of commisioned by
    "has donated": "Q83405216",  # Inverse of donates
    "has sponsored": "Q104231370",  # Inverse of sponsor
    "has subsidiary": "Q105771443",
    "head of government": "Q2285706",
    "headquarters": "Q7540126",
    "owned by": "Q66758185",  # Inverse of owned by
    "ownership": "Q6498684",
    "parent company": "Q1956113",
    "possession": "Q829909",
    # Inverse property label of "represented by"
    "represents": "Q109425297",
    "state administration": "Q12056862",
    "subsidiary": "Q658255",
}

properties_for_power_relations = {
    "affiliation": "P1416",
    # "conferred by": "P1027",  # 6 hits
    "approved by": "P790",
    # "commissioned by": "P88",  # 4 hits
    "distributed by": "P750",
    "donated by": "P1028",
    "founded by": "P112",
    "funder": "P8324",
    "grants": "P5460",
    "has part(s)": "P527",
    # "has subsidiary": "P355",  # 3 hits
    "issued by": "P2378",
    "parent organization": "P749",
    "part of": "P361",
    "partnership with": "P2652",
    "represented by": "P1875",
    "represents": "P1268",
    "sponsor": "P859",
}

items_for_family_relations = {
    "child": "Q29514218",
    "parent": "Q7566",
    # Inverse of child. Canonically parent.
    "parent (inverse)": "Q125693763",
    "sibling of": "Q105123566",  # Inverse of sibling
}

properties_for_family_relations = {
    "child": "P40",
    "relative": "P1038",
    "sibling": "P3373",
    "spouse": "P26",
}

items_for_authorial_relations = {
    "author": "Q482980",
    "author of": "Q65970010",  # Inverse of author
    # Inverse of "contributor of creative work""contributed to work":
    # "Q67089887",
    "creator": "Q2500638",
    "creator of": "Q78522641",
    "distributor of": "Q67508442",
    "has written for": "P6872",
    "publishing group": "Q8869",
    "publishing house": "Q2085381",  # Canonically "publisher".
}

properties_for_authorial_relations = {
    "country": "P17",
    "country of origin": "P495",
    "creator": "P170",
    "location of creation": "P1071",
    "location of formation": "P740",
    "author": "P50",
    "contributor to the creative work or subject": "P767",
    "maintained by": "P126",
    "place of publication": "P291",
    "publication date": "P577",
    "published in": "P1433",
    "publisher": "P123",
    "supervised by": "P7604",
    "time of inception": "P571",  # Canonically inception
    "official website": "P856",
    "point in time": "P585",
    "start time": "P580",
}

items_for_reference_resolution = {
    "abbreviation": "Q102786",
    "full name": "Q1071027",
    "name": "Q82799",
}

properties_for_reference_resolution = {
    "alternative name": "P4970",
    "different from": "P1889",
    "name": "P2561",
    "name in native language": "P1559",
    "nickname": "P1449",
    "pseudonym": "P742",
    "said to be the same as": "P460",
    "short name": "P1813",
    "archive URL": "P1065",
    "author name string": "P2093",
    "synonym": "P5973",
    "URL": "P2699",
}

items_for_occupation_and_specialization = {
    "competence": "Q527091",
    "education": "Q11499482",
    "field of study": "Q1047113",
    "field of work": "Q627436",
    "formal education": "Q11901653",
    "inexperience": "Q108816154",
    "job activity": "Q192581",
    "job title": "Q828803",
    "occupation": "Q12737077",
    "occupation name": "Q116003388",
    "occupation of": "Q66507814",
    "profession": "Q28640",
    "professional certification": "Q16023913",
    "professional qualification": "Q3412758",
    "qualification": "Q4218455",
    "specialized in this field": "Q105476280",  # Inverse of field of
}

properties_for_occupation_and_specialization = {
    "occupation": "P106",
    "academic degree": "P512",
    "educated at": "P69",
    "field of training": "P8258",
    "field of work": "P101",
    "relevant qualification": "P4968",
}

items_related_to_sources_as_data = {
    "data": "Q42848",
    "data base": "Q59138835",
    "data format": "Q494823",
    # Seems to be debated whether this is synonymous  with the space-separated
    # word. The latter is more commonly used."database": "Q8513",
    "data record": "Q2470517",  # Canonically "record"
    "data set": "Q1172284",
    "data source": "Q121566744",
    "document": "Q49848",
    "online database": "Q7094076",
    "online publication": "Q1714118",
    "record": "Q107435521",
}

abstract_items = {
    "contained within": "Q96783935",
    "does not use": "P11484",
    "existence": "Q468777",
    "genre": "Q483394",
    "has member": "Q65971553",
    "is characteristic of": "Q66490689",
    "is not characteristic of": "Q115668717",
    "non-existence": "Q3877969",
    "organization": "Q43229",
    "presence": "Q24255051",
    "science": "Q336",
    "subgenre": "Q18042594",
    "superclass of": "Q66088480",
    "user": "Q613354",  # E.g. of system
    "work of art": "Q386724",  # Canonically "work"
    "work": "Q268378",
}

abstract_properties = {
    "absence": "Q19829125",
    "characteristic of": "P13044",
    # "contains": "P4330",  # 4 hits
    "does not have characteristic": "P6477",
    "does not have part": "P3113",
    "genre": "P136",
    "has characteristic": "P1552",
    # "in opposition to": "P5004",  1 hit
    "including": "P1012",
    "instance of": "P31",
    "inverse label item": "P7087",
    "inverse property": "P1696",
    "language of work or name": "P407",
    "location": "P276",
    "main subject": "P921",
    "opposite of": "P461",
    "subclass of": "P279",
    "subproperty of": "P1647",
    "used by": "P1535",
    "uses": "P2283",
}

journalism_items = {
    "advocacy": "Q2509280",
    "advocacy group": "Q431603",
    "argument": "Q186619",
    "article": "Q191067",
    "book": "Q571",
    "charity": "Q1077064",
    "communication": "Q11024",
    "communications media": "Q340169",
    "conflict": "Q180684",
    "controversy": "Q1255828",
    "convincing": "Q19357393",  # The process
    "creative work": "Q17537576",
    "creative work or creator of work in this genre": "Q68715390",
    "daily newspaper": "Q1110794",
    "document genre": "Q107478770",
    "editorial board": "Q2985386",
    "editorial policy": "Q3240166",
    "editorial team": "Q24354647",
    "educator": "Q974144",
    "experience": "Q164359",
    "ignorance": "Q815577",
    "incompetence": "Q62562253",
    "information": "Q11028",
    "information media": "Q30032147",
    "information resource": "Q37866906",
    "journalism": "Q11030",
    "journalism genre": "Q4894405",
    "journalist": "Q1930187",
    "journalistic report": "Q1030037",  # Canonically: report
    "journalistic reporter": "Q42909",  # Canonically: reporter
    "lobbying": "Q187117",
    "lobbyist": "Q11986654",
    "magazine & newspaper": "Q117208269",
    "mass communication": "Q853710",
    "mass media": "Q11033",
    "media company": "Q1331793",
    "media manipulation": "Q12606852",
    "media profession": "Q58635633",
    "message": "Q628523",
    "news": "Q38926",
    "news media": "Q1193236",
    "news style": "Q2469603",  # I.e. prose style
    "news website": "Q17232649",
    "newspaper": "Q11032",
    "newspaper format": "Q187051",
    "non-fiction": "Q213051",
    "non-fiction literature": "Q27801",
    "non-fiction writer": "Q15980158",
    "online magazine": "Q847906",
    "online newspaper": "Q1153191",
    "periodical": "Q1002697",
    "policy": "Q1156854",
    "popularity": "Q1357284",
    "pressure group": "Q1666019",
    "the process of education": "Q8434",  # Canonically: education
    "protest group": "Q131920868",
    "public administration": "Q31728",
    "public authority": "Q4383245",
    "public institution": "Q1549152",
    "public office": "Q294414",
    "public opinion": "Q17946",
    "publication": "Q732577",
    "publication date": "Q1361758",
    # Person or company responsible for publishing being enacted"
    "publisher": "Q105044823",
    # Canonically "publisher". Person who publishes privately or with their
    # company
    "publisher (person)": "Q2516866",
    "report": "Q10870555",
    "reportage": "Q124922",
    "reporter": "Q11348441",
    "research": "Q42240",
    "social media": "Q202833",
    "social influence": "Q1307337",
    "teaching": "Q352842",
    "topic": "Q26256810",
    "troll": "Q11281",
    "trolling": "Q64737274",
    "type of mass media": "Q121132955",
    "type of publication": "Q39725049",
    "web page": "Q36774",
    "website": "Q35127",
    "writer": "Q36180",
    "written news media": "Q17172633",
    "written work": "Q47461344",
}

What should the canonical graph population look like? What query would it imply?  

One important high-level thing to keep in mind is that for bad-labeled seeds, we are skeptical of what they relate to, but for good-labeled items, we might want them to "endorsed by" or "reviewed by" sources.

For now, I'm working under the assumption that it should be arbitrary who is the subject and who is the object of a relation wrt. adding labels later. I.e. I expect to join the outcome dataset with the labelled dataset by joining on IDs.  

There are A LOT of decisions to be made here. For example, an early run yielded way too many "employed-by" relations. For now, I will adopt the idea that `single people participate in the seed dataset for label propagation, but we are not interested in scraping single people (non-organizations) during graph extension`.

Note an important conceptual divergence here: Our neighbors-datasets are built over the Subject-Relation-Object triple, and hence, they are no longer indexed by a single source, and hence should not hold information about these. Therefore, we will need to also maintain a dataframe of information over items.

In [22]:
# Greedy properties
properties_for_appraisal = {
    "assessment": "P5021",
    "complies with": "P5009",
    "endorsed by": "P8001",
    # "review of": "P6977",
    # "reviewed by": "P4032",
}
properties_for_citation = {
    # "inferred from": "P3452",
    # "value suitable for "stated in" (P248) when property is used in
    # references to refer to the primary publication/website/database
    # associated with the property"
    # "applicable 'stated in' value": "P9073",
    "cites work": "P2860",
    # "described by source": "P1343",
    "directs readers to": "P13269",
    "quotation": "P1683",  # I.e. of a source
    "quotation or excerpt": "P7081",
    "quotes work": "P6166",
    # "stated in": "P248",
    # That is, when a text asserts that another text asserts that a
    # third text says something; to be used with the qualifier P248
    "stated in source according to": "P12132",
}
properties_for_ideology = {
    "member of political party": "P102",
}

properties_for_personnel = {
    "chairperson": "P488",
    "director / manager": "P1037",
    "position holder": "P1308",
    "board member": "P3320",
    "chief executive officer": "P169",
    "chief operating officer": "P1789",
    "editor": "P98",
    "editor-in-chief": "P5769",
    "employer": "P108",
    "general secretary": "P3975",
    "head of government": "P6",
    "member of": "P463",
    # some or all members of this group share the occupation
    "owned by": "P127",
    "owner of": "P1830",
    # "position held": "P39",
    # "position held by head of the organization": "P2388",
}

properties_for_power_relations = {
    "affiliation": "P1416",
    # "conferred by": "P1027",  # 6 hits
    "approved by": "P790",
    # "commissioned by": "P88",  # 8 hits
    "distributed by": "P750",
    "donated by": "P1028",
    "founded by": "P112",
    "funder": "P8324",
    "grants": "P5460",
    "has part(s)": "P527",
    # "has subsidiary": "P355",  # 3 hits
    "issued by": "P2378",
    "parent organization": "P749",
    "part of": "P361",
    "partnership with": "P2652",
    "represented by": "P1875",
    # "represents": "P1268",  # Bad hits
    "sponsor": "P859",
}

properties_for_family_relations = {
    "child": "P40",
    "relative": "P1038",
    "sibling": "P3373",
    "spouse": "P26",
    "father": "P22",
    "mother": "P25",
}

properties_for_authorial_relations = {
    # "country": "P17",
    # "country of origin": "P495",
    # "creator": "P170",  # 4 hits
    # "location of creation": "P1071",
    # "location of formation": "P740",
    # "author": "P50",  # Bad hits
    "contributor to the creative work or subject": "P767",
    "maintained by": "P126",
    # "place of publication": "P291",  # Geographical
    "published in": "P1433",
    # "publisher": "P123",  # Bad hits
    "supervised by": "P7604",
    # "official website": "P856",
}

# properties_for_occupation_and_specialization = {
#     # "occupation": "P106",
#     # "educated at": "P69",
#     # "relevant qualification": "P4968",  # ??
# }

abstract_properties = {
    # "contains": "P4330",  # 4 hits
    # "genre": "P136",
    # "in opposition to": "P5004",
    "including": "P1012",
    # "used by": "P1535",  # Bad hits
    # "uses": "P2283",  # Bad hits
}


# Non-greedy properties; abandoned.
# one_hop_endorsement_properties = {
#     "endorsed by": "P8001",
#     "reviewed by": "P4032",
#     "directs readers to": "P13269",
#     "quotation": "P1683",  # I.e. of a source
#     "approved by": "P790",
#     "supervised by": "P7604",
# }

# one_hop_positional_properties = {
#     "member of political party": "P102",
#     "chairperson": "P488",
#     "director / manager": "P1037",
#     "position holder": "P1308",
#     "board member": "P3320",
#     "chief executive officer": "P169",
#     "chief operating officer": "P1789",
#     "editor": "P98",
#     "editor-in-chief": "P5769",
#     "general secretary": "P3975",
#     "head of government": "P6",
#     "member of": "P463",
#     "owned by": "P127",
#     "owner of": "P1830",
#     "position held": "P39",
#     "position held by head of the organization": "P2388",
# }

# one_hop_power_properties = {
#     "affiliation": "P1416",
#     "conferred by": "P1027",
#     "commissioned by": "P88",
#     "distributed by": "P750",
#     "donated by": "P1028",
#     "founded by": "P112",
#     "funder": "P8324",
#     "grants": "P5460",
#     "has part(s)": "P527",
#     "has subsidiary": "P355",
#     "issued by": "P2378",
#     "parent organization": "P749",
#     "part of": "P361",
#     "partnership with": "P2652",
#     "represented by": "P1875",
#     "represents": "P1268",
#     "sponsor": "P859",
# }

# one_hop_authorship_properties = {
#     "maintained by": "P126",
#     "published in": "P1433",
#     "publisher": "P123",
# }

# one_hop_reference_properties = {
#     "alternative name": "P4970",
#     "nickname": "P1449",
#     "pseudonym": "P742",
#     "said to be the same as": "P460",
#     "synonym": "P5973",
# }

# PROPERTIES = dict([
#     i
#     for d in [
#         one_hop_endorsement_properties,
#         one_hop_positional_properties,
#         one_hop_power_properties,
#         one_hop_authorship_properties,
#     ]
#     for i in d.items()
#     # "endorsement": one_hop_endorsement_properties,
#     # "positional": one_hop_positional_properties,
#     # "power": one_hop_power_properties,
#     # "authorship": one_hop_authorship_properties,
#     # "reference": one_hop_reference_properties,
# ])


GREEDY_PROPERTIES = dict(
    [  # Absolutely all relevant properties
        i
        for d in [
            properties_for_appraisal,
            properties_for_citation,
            properties_for_ideology,
            properties_for_personnel,
            properties_for_power_relations,
            properties_for_family_relations,
            properties_for_authorial_relations,
            abstract_properties,
        ]
        for i in d.items()
    ]
)

# Assign specific batch-sizes to properties known to timeout. Use the
# value 0 to skip. That's a rough mechanism that should only be used in
# an exploratory phase.
CUSTOM_PROPERTY_BATCH_SIZES = {
    "employer": 2,  # 5
    "published in": 75,
}

"""
These entities invariably provoke timeouts regardless of batch size.
Contains:
    * The two major US and UK political parties
    * The class of independent politicians
    * NetFlix
    * The class of news agencies
    * TikTok
    * The New York Times
    * The (US) Presidential Materials Division
    * List of programs broadcast by Investigation Discovery
"""
EXCLUSIONS = [
    "Q29468",
    "Q29552",
    "Q327591",
    "Q9626",
    "Q9630",
    "Q907311",
    "Q192283",
    "Q48938223",
    "Q9684",
    "Q59661092",
    "Q59661072",
    "Q28024847",
]


# At this point, we can try to remove all class instances rather than
# real instances by hand.
CLASS_TYPE_SOURCES_TO_DROP = [
    "public note",
    "newspaper",
    "mass media",
    "coroner",
    "family foundation",
    "campaign website",
    "blog post",
    "campaign advertising",
    "article",
    "news agency",
    "press release",
    "bumper sticker",
    "blog",
    "independent politician",
    "tweet",
    "news article",
    "staff writer",
    "television advertisement",
    "river source",
    "website",
    "navy",
    "coast guard",
    "column",
]

When propagating the graph, a layer of complexity emerges in the targets of queries.

* For the most part, a high-level distinction between human and nonhuman entities is useful.
* All four possible relations between two such entities need to be considered. E.g. nonhumans can be affiliated to nonhumans. Nonhumans can be owned by humans. Humans can be members of nonhumans. Humans can be in partnership with humans.
* TEMP: Perhaps the most tricky part: There are ways that nonhumans can relate to humans such that it becomes critical to filter whether the humans in question are relevant. For example, the "employer" relation can target a huge amount of potentially irrelevant humans.

For the next part, it becomes important to split our dataset into humans and nonhumans.

In [24]:
wikifake_humans_df = wikifake_df[wikifake_df["sourceCategory"] == "person"]
wikifake_nonhumans_df = wikifake_df[wikifake_df["sourceCategory"] != "person"]

In [ ]:
ACCEPTED_HUMAN_ITEMS = {
    "activist": "Q15253558",
    "business magnate": "Q911554",
    "celebrity": "Q211236",  # subclasses: YouTuber
    "content creator": "Q109459317",
    "economist": "Q188094",
    "functionary": "Q1328323",
    "jurist": "Q185351",
    "knowledge worker": "Q1914017",  # Subclasses: scientist
    "leader": "Q1251441",  # Subclasses: oligarch
    "manager": "Q2462658",  # Subclasses: spiritual leader
    "media professional": "Q15980804",  # Subclasses: media personality, journalist
    "official": "Q599151",
    "politician": "Q82955",  # subclasses: president, statesperson
    "scholar": "Q20826540",
    "researcher": "Q1650915",
}

ACCEPTED_NONHUMAN_ITEMS = {
    "advocacy group": "Q431603",
    "broadcaster": "Q15265344",
    "business and professional associations, unions": "Q130370871",  # subclasses: trade unions,
    "committee": "Q865588",
    # "company": "Q783794",  # subclasses: media company
    "council": "Q4358176",
    "educational organization": "Q5341295",
    "GLAM": "Q1030034",  # Galleries, libraries, archives and museums
    "media company": "Q1331793",
    "medical organization": "Q4287745",
    "nonprofit organization": "Q163740",  # Subclasses: Foundation, NGO, philanthropic organization
    # "organization": "Q43229",  # Subclasses: non-profit organization, NGO, think tank, lobbying org, public company, committees, councils, political institution, educational organization, university, GLAM (galleries, libraries, archives and museums), broadcaster, medical organization,
    "podcast show": "Q24634210",
    "political institution": "Q18810687",
    "periodical": "Q1002697",
    # "publication": "Q732577",  # Subclasses: website
    "special interest group": "Q2504350",
    "think tank": "Q155271",
    "university": "Q3918",
    "website": "Q35127",
}

nonhuman_nonhuman_relations = {
    # "cites work": "P2860",
    # "directs readers to": "P13269",
    # "quotation": "P1683",  # I.e. of a source
    # "quotation or excerpt": "P7081",
    # "quotes work": "P6166",
    "affiliation": "P1416",
    # "conferred by": "P1027",  # 6 hits
    "approved by": "P790",
    # "commissioned by": "P88",  # Bad hits
    # "distributed by": "P750",
    "donated by": "P1028",
    "has part(s)": "P527",
    "has subsidiary": "P355",  # 3 hits
    "issued by": "P2378",
    "parent organization": "P749",
    "part of": "P361",
    "partnership with": "P2652",
    # "represented by": "P1875",
    # "represents": "P1268",  # Mostly just hit sports teams
    "sponsor": "P859",
    "maintained by": "P126",
    # "published in": "P1433",
    # "publisher": "P123",  # Bad hits
    # "contains": "P4330",  # Bad hits
    "including": "P1012",
    "member of": "P463",
}

nonhuman_human_relations = {
    "conferred by": "P1027",  # 6 hits
    "approved by": "P790",
    "commissioned by": "P88",  # 8 hits
    "distributed by": "P750",
    "donated by": "P1028",
    "issued by": "P2378",
    "partnership with": "P2652",
    "represented by": "P1875",
    "sponsor": "P859",
    "contributor to the creative work or subject": "P767",
    "maintained by": "P126",
    "publisher": "P123",  # Bad hits
    "supervised by": "P7604",
    "chairperson": "P488",
    "director / manager": "P1037",
    "board member": "P3320",
    "chief executive officer": "P169",
    "chief operating officer": "P1789",
    "editor": "P98",
    "editor-in-chief": "P5769",
    "general secretary": "P3975",
    # "head of government": "P6",
    "owned by": "P127",
    "creator": "P170",  # 4 hits
    # "author": "P50",  # Bad hits
}

human_nonhuman_relations = {
    # "affiliation": "P1416",
    "funder": "P8324",
    "partnership with": "P2652",
    "sponsor": "P859",
    "member of political party": "P102",
    # "employer": "P108",  # THIS ONE IS TRICKY!
}

human_human_relations = {
    "affiliation": "P1416",
    "partnership with": "P2652",
    "sponsor": "P859",
    "supervised by": "P7604",
}


RELATIONS_DICT = {
    "nonhuman_nonhuman": nonhuman_nonhuman_relations,
    "nonhuman_human": nonhuman_human_relations,
    "human_nonhuman": human_nonhuman_relations,
    "human_human": human_human_relations,
}

HUMAN_PROPERTIES_DICT = {"occupation": "P106", "relevant qualification": "P4968"}

In [ ]:
# We now extend to a dataframe that holds both seed and non-seed
# sources. Keeping a saved "wikifake" file is useful, but in the
# following code, it is more logically clear to keep a single dataset of
# sources from which wikidata can be retrieved at anytime by filtering
# for items with label (maps).


def rename_triple_roles_to_source(
    df, role: Union[Literal["subject"], Literal["object"]]
):
    """
    For a dataframe with columns with a "subject" or "object" prefix,
    replace it with "source".
    """
    return df.rename(columns={col: col.replace(role, "source") for col in df.columns})


def _perform_moded_hop(
    mode: Union[
        Literal["nonhuman_nonhuman"],
        Literal["nonhuman_human"],
        Literal["human_nonhuman"],
        Literal["human_human"],
    ],
    df: DatasetObject,
    accepted_items_dict: dict[str, str],
    start_idx: int,
    known_stalls: dict[int, dict[str, set[str]]],
    hop: int,
    relations_dict: dict[str, dict[str, str]],
    batch_size: int,
    name: str,
    exclusions: Iterable[str],
    custom_batch_sizes: dict[str, dict[str, int]],
    stalling_relations_filename: Optional[str] = None,
    human_properties_dict: Optional[dict[str, str]] = None,
    **kwargs,
):
    print(f"Performing partial hop: {hop}: {mode}.")
    if mode not in known_stalls[hop]:
        known_stalls[hop][mode] = set()
    if mode not in custom_batch_sizes.keys():
        custom_batch_sizes[mode] = dict()
    df = df[~df["source"].isin(exclusions)][start_idx:]
    relations_to_skip = known_stalls[hop][mode]
    relations_for_mode = relations_dict[mode]
    custom_batch_sizes_for_mode = custom_batch_sizes[mode]
    res_df = pd.DataFrame()
    for relation_name, relation_id in relations_for_mode.items():
        if relation_name in custom_batch_sizes_for_mode.keys():
            absolute_batch_size = custom_batch_sizes_for_mode[relation_name]
        else:
            absolute_batch_size = batch_size
        if relation_name in relations_to_skip or absolute_batch_size == 0:
            print(f"Skipping {relation_name}.")
            continue
        dataset_path = in_neighbors_dir(
            name
            + "_"
            + mode
            + "_"
            # Try to avoid the Windows MAXPATH
            + relation_name.replace("/", "_")[:14]  # Catches one trickster
            + "_hop_"
            + str(hop)
        )
        if os.path.exists(in_data_dir(dataset_path)):
            print(f"Found existing saved dataset for {dataset_path}.")
            res = datasets.Dataset.load_from_disk(in_data_dir(dataset_path))
        else:
            print(f"Querying with {relation_name}...")
            absolute_kwargs = {
                "batch_size": absolute_batch_size,
                "id_cols": ["subject", "object", "relation"],
                "accepted_items_dict": accepted_items_dict,
                **kwargs,
            }
            match mode:
                case "nonhuman_nonhuman":
                    res = construct_dataset_from_queries(
                        df,
                        g.for_neighbor_organizations,
                        dataset_path,
                        accepted_properties_dict={relation_name: relation_id},
                        **absolute_kwargs,
                        bidirectional=True,
                    )
                case "nonhuman_human":
                    res = construct_dataset_from_queries(
                        df,
                        g.for_neighbor_humans,
                        dataset_path,
                        accepted_relations_dict={relation_name: relation_id},
                        human_properties_dict=human_properties_dict,
                        **absolute_kwargs,
                        bidirectional=False,
                    )
                case "human_nonhuman":
                    res = construct_dataset_from_queries(
                        df,
                        g.for_neighbor_organizations,
                        dataset_path,
                        accepted_properties_dict={relation_name: relation_id},
                        **absolute_kwargs,
                        bidirectional=False,
                    )
                case "human_human":
                    res = construct_dataset_from_queries(
                        df,
                        g.for_neighbor_humans,
                        dataset_path,
                        accepted_relations_dict={relation_name: relation_id},
                        human_properties_dict=human_properties_dict,
                        **absolute_kwargs,
                        bidirectional=True,
                    )
        if len(res):
            _res_df = res.to_pandas().groupby(["subject", "relation", "object"]).first()
            print(f"'{relation_name}' had {len(_res_df)} hits!")
            res_df = pd.concat([res_df, _res_df])
        else:
            print(f"Propagation for '{relation_name}' stalled at hop {hop}!")
            # This shouldn't happen, but this check can fail if multiple
            # sequential runs of the function are not properly
            # synchronized.
            known_stalls[hop][mode].add(relation_name)
            if stalling_relations_filename:
                with open(stalling_relations_filename, "wb") as f:
                    pickle.dump(known_stalls, f)
    return res_df, known_stalls


def apply_filters_to_sro_df(df: DatasetObject, batch_size=200) -> DatasetObject:
    input_len = len(df)
    if input_len:
        ids_to_include = set()
        for idx in tqdm(range(0, input_len, batch_size), desc="Filtering results..."):
            res = execute_sparql_query(g.apply_filters(df[idx : idx + batch_size]))
            if len(res):
                ids_to_include.update(set(res["id"].str.rpartition("/")[2]))
        return df[
            (df["subject"].isin(ids_to_include) | df["object"].isin(ids_to_include))
        ]
    else:
        return df


def multihop_propagation(
    nonhumans_df: DatasetObject,
    humans_df: DatasetObject,
    accepted_human_items_dict: dict[str, str],
    accepted_nonhuman_items_dict: dict[str, str],
    human_properties_dict: dict[
        str, str
    ],  # The ways humans may relate to accepted classes, e.g. "occupation"
    relations_dict: dict[str, dict[str, str]],
    name: str,
    exclusions: list[str] = [],
    hops=1,
    batch_size=300,
    stalling_properties_filename="_stalling_properties",
    custom_batch_sizes: Optional[dict[str, dict[str, int]]] = None,
    purge_dir=False,
    **kwargs,
):
    """
    For a chosen number of hops, find SRO-triples while updating the
    source datasets.

    Note that in this function, we use single properties at a time but
    high batchsizes.

    The function is designed to be run end-to-end; the associated saving
    and skipping behavior is brittle, and should only be used
    exploratorily:
        A pickled dict is kept which records known hops in which a
        property has no hits ("stalls"). Make sure to delete this file
        if you are changing querying behavior.
    """
    if purge_dir and os.path.exists(in_data_dir(in_neighbors_dir(name))):
        if input(f'Are you sure you want to purge the "{name}" directory? (y)') == "y":
            shutil.rmtree(in_data_dir(in_neighbors_dir(name)))
            print("Directory purged.\n")
            if os.path.exists(stalling_properties_filename) and (
                input("Do you also want to remove the stalling_properties_file? (y)")
                == "y"
            ):
                os.remove(stalling_properties_filename)
    if not os.path.exists(in_data_dir(in_neighbors_dir(name))):
        os.mkdir(in_data_dir(in_neighbors_dir(name)))
    if custom_batch_sizes is None:
        custom_batch_sizes = {
            "nonhuman_nonhuman": dict(),
            "nonhuman_human": dict(),
            "human_nonhuman": dict(),
            "human_human": dict(),
        }
    nonhumans_start_idx = 0
    humans_start_idx = 0
    if os.path.exists(stalling_properties_filename):
        print(f"Found existing known-stalls file.")
        with open(stalling_properties_filename, "rb") as f:
            known_stalls = pickle.load(f)
    else:
        known_stalls = {}
    nonhuman_neighbors_df = pd.DataFrame()
    human_neighbors_df = pd.DataFrame()
    for hop in range(hops):
        len_nonhuman_neighbors_before_hop = len(nonhuman_neighbors_df)
        len_human_neighbors_before_hop = len(human_neighbors_df)
        print(f"\nPerforming hop {hop}...")
        if hop not in known_stalls.keys():
            known_stalls[hop] = dict()
        args = (
            hop,
            relations_dict,
            batch_size,
            # The leaf directory has the same name as the model
            os.path.join(name, name),
            exclusions,
            custom_batch_sizes,
            stalling_properties_filename,
        )
        nonhuman_nonhuman_neighbors_df, known_stalls = _perform_moded_hop(
            "nonhuman_nonhuman",
            nonhumans_df,
            accepted_nonhuman_items_dict,
            nonhumans_start_idx,
            known_stalls,
            *args,
            **kwargs,
        )
        nonhuman_human_neighbors_df, known_stalls = _perform_moded_hop(
            "nonhuman_human",
            nonhumans_df,
            accepted_human_items_dict,
            nonhumans_start_idx,
            known_stalls,
            *args,
            human_properties_dict=human_properties_dict,
            **kwargs,
        )
        human_nonhuman_neighbors_df, known_stalls = _perform_moded_hop(
            "human_nonhuman",
            humans_df,
            accepted_nonhuman_items_dict,
            humans_start_idx,
            known_stalls,
            *args,
            **kwargs,
        )
        human_human_neighbors_df, known_stalls = _perform_moded_hop(
            "human_human",
            humans_df,
            accepted_human_items_dict,
            humans_start_idx,
            known_stalls,
            *args,
            human_properties_dict=human_properties_dict,
            **kwargs,
        )
        # Update indices, concatenate dfs, update node_df
        print(f"Finished hop. Population and concatenating datasets...")
        nonhumans_start_idx = len(nonhumans_df)
        humans_start_idx = len(humans_df)
        if os.path.exists(
            path_ := in_data_dir(
                in_neighbors_dir(
                    name + "_nonhuman_neighbors_hop_" + str(hop) + "_filtered"
                )
            )
        ):
            nonhuman_neighbors_df = datasets.Dataset.load_from_disk(path_).to_pandas()
        else:
            nonhuman_neighbors_df = apply_filters_to_sro_df(
                pd.concat(
                    [
                        _df.reset_index()
                        for _df in [
                            nonhuman_nonhuman_neighbors_df,
                            human_nonhuman_neighbors_df,
                            nonhuman_neighbors_df,
                        ]
                        if len(_df)
                    ],
                ).drop_duplicates()
            )
            # I end up with duplicate index columns
            for col in ["__index_level_0__", "level_0"]:
                if col in nonhuman_neighbors_df.columns:
                    nonhuman_neighbors_df.drop(col, axis=1, inplace=True)

            datasets.Dataset.from_pandas(
                nonhuman_neighbors_df.astype(str)
            ).save_to_disk(path_)
        if os.path.exists(
            path_ := in_data_dir(
                in_neighbors_dir(
                    name + "_human_neighbors_hop_" + str(hop) + "_filtered"
                )
            )
        ):
            human_neighbors_df = datasets.Dataset.load_from_disk(path_).to_pandas()
        else:
            human_neighbors_df = apply_filters_to_sro_df(
                pd.concat(
                    [
                        _df.reset_index()
                        for _df in [
                            nonhuman_human_neighbors_df,
                            human_human_neighbors_df,
                            human_neighbors_df,
                        ]
                        if len(_df)
                    ]
                ).drop_duplicates()
            )
            for col in ["__index_level_0__", "level_0"]:
                if col in human_neighbors_df.columns:
                    human_neighbors_df.drop(col, axis=1, inplace=True)
            datasets.Dataset.from_pandas(human_neighbors_df.astype(str)).save_to_disk(
                path_
            )
        nonhuman_hits_on_hop = (
            len(nonhuman_neighbors_df) - len_nonhuman_neighbors_before_hop
        )
        human_hits_on_hop = len(human_neighbors_df) - len_human_neighbors_before_hop
        if nonhuman_hits_on_hop:
            nonhumans_df = populate_nodes_df_from_sro_df(
                nonhuman_neighbors_df, nonhumans_df
            )
        if human_hits_on_hop:
            humans_df = populate_nodes_df_from_sro_df(human_neighbors_df, humans_df)
        if nonhuman_hits_on_hop or human_hits_on_hop:
            print(
                f"Hop had a total of {human_hits_on_hop + nonhuman_hits_on_hop} hits.\n"
            )
        else:
            print(f"Graph population stalled universally at hop {hop}!")
            break
    print("Done! Finalizing datasets.")
    # There is sufficient symmetry in the queries that we need to drop
    # duplicates.
    neighbors_df = (
        pd.concat([nonhuman_neighbors_df, human_neighbors_df])
        .astype(str)
        .drop_duplicates()
        .reset_index(drop=True)
    )
    nodes_df = (
        pd.concat([nonhumans_df, humans_df])
        .astype(str)
        .drop_duplicates()
        .reset_index(drop=True)
    )
    datasets.Dataset.from_pandas(neighbors_df).save_to_disk(
        in_data_dir(in_neighbors_dir(name) + "_final")
    )
    datasets.Dataset.from_pandas(nodes_df).save_to_disk(
        in_data_dir(f"wikifake_plus_{name}")
    )
    return neighbors_df, nodes_df


nodes_df = wikifake_df
nodes_df = nodes_df[~nodes_df["sourceLabel"].isin(CLASS_TYPE_SOURCES_TO_DROP)]

CUSTOM_BATCH_SIZES = {}
CUSTOM_BATCH_SIZES["nonhuman_nonhuman"] = {
    "affiliation": 100,
    "published in": 100,
    # "publisher": 200,
    # "part of": 200,
    "maintained by": 2,
    "member of": 25,
    # "including": 200,
    "has part(s)": 100,
    "has subsidiary": 100,
    # "issued by": 200,
    # "parent organization": 200,
}

# three_hop_neighbors_with_universities_df, three_hop_with_universities_wikifake_df = (
#     multihop_propagation(
#         wikifake_nonhumans_df,
#         wikifake_humans_df,
#         accepted_nonhuman_items_dict=ACCEPTED_NONHUMAN_ITEMS,
#         accepted_human_items_dict=ACCEPTED_HUMAN_ITEMS,
#         human_properties_dict=HUMAN_PROPERTIES_DICT,
#         # exclusions=EXCLUSIONS,
#         relations_dict=RELATIONS_DICT,
#         name="three_hop_with_universities_neighbors",
#         hops=3,
#         custom_batch_sizes=CUSTOM_BATCH_SIZES,
#         behavior="raise",
#         # overwrite=True,
#         purge_dir=False,
#         stalling_properties_filename="_stalling_properties_three_hop_with_universities",
#     )
# )

In [34]:
accepted_nonhuman_items_no_universities = deepcopy(ACCEPTED_NONHUMAN_ITEMS)
accepted_nonhuman_items_no_universities.pop("educational organization")
accepted_nonhuman_items_no_universities.pop("university")

two_hop_neighbors_df, two_hop_wikifake_df = multihop_propagation(
    wikifake_nonhumans_df,
    wikifake_humans_df,
    accepted_nonhuman_items_dict=accepted_nonhuman_items_no_universities,
    accepted_human_items_dict=ACCEPTED_HUMAN_ITEMS,
    human_properties_dict=HUMAN_PROPERTIES_DICT,
    # exclusions=EXCLUSIONS,
    relations_dict=RELATIONS_DICT,
    name="two_hop_neighbors",
    hops=2,
    custom_batch_sizes=CUSTOM_BATCH_SIZES,
    behavior="raise",
    # overwrite=True,
    purge_dir=False,
    stalling_properties_filename="_stalling_properties_two_hop",
)

Found existing known-stalls file.

Performing hop 0...
Performing partial hop: 0: nonhuman_nonhuman.
Found existing saved dataset for neighbor_datasets\two_hop_neighbors\two_hop_neighbors_nonhuman_nonhuman_affiliation_hop_0.
'affiliation' had 1054 hits!
Found existing saved dataset for neighbor_datasets\two_hop_neighbors\two_hop_neighbors_nonhuman_nonhuman_approved by_hop_0.
'approved by' had 1 hits!
Found existing saved dataset for neighbor_datasets\two_hop_neighbors\two_hop_neighbors_nonhuman_nonhuman_donated by_hop_0.
'donated by' had 1 hits!
Found existing saved dataset for neighbor_datasets\two_hop_neighbors\two_hop_neighbors_nonhuman_nonhuman_has part(s)_hop_0.
'has part(s)' had 537 hits!
Found existing saved dataset for neighbor_datasets\two_hop_neighbors\two_hop_neighbors_nonhuman_nonhuman_has subsidiary_hop_0.
'has subsidiary' had 534 hits!
Skipping issued by.
Found existing saved dataset for neighbor_datasets\two_hop_neighbors\two_hop_neighbors_nonhuman_nonhuman_parent organi

Constructing dataset from queries...: 100%|██████████| 694/694 [06:11<00:00,  1.87it/s] 


Saving the dataset (0/1 shards):   0%|          | 0/148 [00:00<?, ? examples/s]

'maintained by' had 59 hits!
Querying with including...
Performing dataset construction from scratch.


Constructing dataset from queries...: 100%|██████████| 12/12 [00:08<00:00,  1.43it/s]


...

Filtering results...: 100%|██████████| 16/16 [00:37<00:00,  2.32s/it]


Saving the dataset (0/1 shards):   0%|          | 0/3001 [00:00<?, ? examples/s]

Hop had a total of 18452 hits.

Done! Finalizing datasets.


Saving the dataset (0/1 shards):   0%|          | 0/26853 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25446 [00:00<?, ? examples/s]

In [ ]:
accepted_nonhuman_items_no_universities = deepcopy(ACCEPTED_NONHUMAN_ITEMS)
accepted_nonhuman_items_no_universities.pop("educational organization")
accepted_nonhuman_items_no_universities.pop("university")

four_hop_neighbors_df, four_hop_wikifake_df = multihop_propagation(
    wikifake_nonhumans_df,
    wikifake_humans_df,
    accepted_nonhuman_items_dict=accepted_nonhuman_items_no_universities,
    accepted_human_items_dict=ACCEPTED_HUMAN_ITEMS,
    human_properties_dict=HUMAN_PROPERTIES_DICT,
    # exclusions=EXCLUSIONS,
    relations_dict=RELATIONS_DICT,
    name="five_hop_neighbors",
    hops=4,
    custom_batch_sizes=CUSTOM_BATCH_SIZES,
    behavior="raise",
    # overwrite=True,
    purge_dir=False,
    stalling_properties_filename="_stalling_properties_five_hop",
)

Found existing known-stalls file.

Performing hop 0...
Performing partial hop: 0: nonhuman_nonhuman.
Found existing saved dataset for neighbor_datasets\five_hop_neighbors\five_hop_neighbors_nonhuman_nonhuman_affiliation_hop_0.
'affiliation' had 1034 hits!
Found existing saved dataset for neighbor_datasets\five_hop_neighbors\five_hop_neighbors_nonhuman_nonhuman_approved by_hop_0.
'approved by' had 1 hits!
Found existing saved dataset for neighbor_datasets\five_hop_neighbors\five_hop_neighbors_nonhuman_nonhuman_donated by_hop_0.
'donated by' had 2 hits!
Found existing saved dataset for neighbor_datasets\five_hop_neighbors\five_hop_neighbors_nonhuman_nonhuman_has part(s)_hop_0.
'has part(s)' had 531 hits!
Found existing saved dataset for neighbor_datasets\five_hop_neighbors\five_hop_neighbors_nonhuman_nonhuman_has subsidiary_hop_0.
'has subsidiary' had 534 hits!
Skipping issued by.
Found existing saved dataset for neighbor_datasets\five_hop_neighbors\five_hop_neighbors_nonhuman_nonhuman_p

Saving the dataset (0/1 shards):   0%|          | 0/90349 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/55623 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
if not RUN_ALL:
    # five_hop_neighbors_df = datasets.load_from_disk(
    #     in_data_dir("five_hop_neighbors_final")
    # ).to_pandas().drop_duplicates(["subject", "relation", "object"]).reset_index()
    # wikifake_plus_five_hop_neighbors = datasets.load_from_disk(
    #     in_data_dir("wikifake_plus_five_hop_neighbors")
    # )
    three_hop_with_universities_neighbors_df = datasets.load_from_disk(
        in_data_dir("three_hop_neighbors_with_universities_final")
    ).to_pandas().drop_duplicates(["subject", "relation", "object"]).reset_index()
    wikifake_plus_three_hop_with_universities_neighbors = datasets.load_from_disk(in_data_dir("wikifake_plus_three_hop_with_universities_neighbors")
    ).to_pandas().reset_index()
    wikifake_df = datasets.load_from_disk(in_data_dir("wikifake")).to_pandas()
    four_hop_neighbors_df = datasets.load_from_disk(
        in_data_dir("four_hop_neighbors_final")
    ).to_pandas().drop_duplicates(["subject", "relation", "object"]).reset_index()
    wikifake_plus_four_hop_neighbors = datasets.load_from_disk(
        in_data_dir("wikifake_plus_four_hop_neighbors")
    )

Some summary statistics.

In [ ]:
def print_df_summary_statistics(df):
    print("Size:")
    print(len(df))
    print("Proportion of relationLabels:")
    print(df.value_counts(subset="relationLabel", normalize=True), end="\n\n")
    print("Most common items, absolute value:")
    print(
        pd.concat(
            [
                df["subjectLabel"],
                df["objectLabel"],
            ]
        ).value_counts()[:20],
        end="\n\n",
    )
    print("Most common items, normalized")
    print(
        pd.concat(
            [
                df["subjectLabel"],
                df["objectLabel"],
            ]
        ).value_counts(
            normalize=True
        )[:20],
        end="\n\n",
    )
    print("Most common subject-relation pairs:")
    print(df.value_counts(subset=["subjectLabel", "relationLabel"])[:20], end="\n\n")

    print("Most common relation-object pairs:")
    print(df.value_counts(subset=["relationLabel", "objectLabel"])[:20], end="\n\n")

    for rel in df["relationLabel"].unique():
        print(f"Subjects of {rel}:")
        print(
            df[df["relationLabel"] == rel].value_counts(subset=["subjectLabel"])[:3],
            end="\n\n",
        )

        print(f"Objects of {rel}:")
        print(
            df[df["relationLabel"] == rel].value_counts(subset=["objectLabel"])[:3],
            end="\n\n",
        )

    print(f"Mean number of relations:")
    print(
        (
            df.groupby(
                ["subjectLabel", "relationLabel", "objectLabel"]
            ).count()
            / df["subjectLabel"].nunique()
        ),
        sep="\n\n",
    )
    print(f"Median number of relations:")
    print(
        df.groupby(["subjectLabel", "relationLabel"])
        .size()
        .groupby("relationLabel")
        .median()
    )

print_df_summary_statistics(four_hop_neighbors_df)
print("\n\n\n")
print_df_summary_statistics(three_hop_with_universities_neighbors_df)

Size:
61297
Proportion of relationLabels:
relationLabel
member of                                      0.628644
part of                                        0.067018
affiliation                                    0.062793
parent organization                            0.054734
has part(s)                                    0.043053
child organization/unit                        0.036755
chairperson                                    0.027734
partnership with                               0.015270
board member                                   0.013231
director / manager                             0.012219
sponsor                                        0.009788
editor                                         0.005433
chief executive officer                        0.005384
maintained by                                  0.005074
editor-in-chief                                0.004242
owned by                                       0.002822
creator                                        0

In [ ]:
print(len(wikifake_plus_three_hop_with_universities_neighbors) - len(wikifake_df))
print(len(wikifake_plus_four_hop_neighbors) - len(wikifake_df))

92439
46349


In [ ]:
datasets.Dataset.from_pandas(four_hop_neighbors_df).save_to_disk(
    in_data_dir("four_hop_neighbors_final")
)
datasets.Dataset.from_pandas(four_hop_wikifake_df).save_to_disk(
    in_data_dir("wikifake_plus_four_hop_neighbors")
)

Saving the dataset (0/1 shards):   0%|          | 0/90349 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/55623 [00:00<?, ? examples/s]

In [ ]:
# wikifake_plus_four_hop_neighbors_df = datasets.Dataset.load_from_disk(
#     in_data_dir("wikifake_plus_four_hop_neighbors")
# ).to_pandas()

# four_hop_neighbors_df = datasets.Dataset.load_from_disk(in_data_dir("four_hop_neighbors_final")).to_pandas()

wikifake_plus_three_hop_neighbors_df = datasets.Dataset.load_from_disk(
    in_data_dir("wikifake_plus_three_hop_with_universities_neighbors")
).to_pandas()

three_hop_neighbors_with_universities_df = datasets.Dataset.load_from_disk(in_data_dir("three_hop_neighbors_with_universities_final")).to_pandas()

#### Investigating the resulting relations

In [ ]:
def inherit_categories_from_seed_df(sro_df, seed_df):
    sro_df["subjectCategory"] = sro_df.merge(
        seed_df, how="left", left_on="subject", right_on="source"
    )["sourceCategory"]

    sro_df["objectCategory"] = sro_df.merge(
        seed_df, how="left", left_on="object", right_on="source"
    )["sourceCategory"]
    return sro_df


three_hop_neighbors_with_universities_df = inherit_categories_from_seed_df(
    three_hop_neighbors_with_universities_df, wikifake_plus_three_hop_neighbors_df
)

# two_hop_neighbors_with_universities_df = inherit_categories_from_seed_df(
    # two_hop_neighbors_with_universities_df, wikifake_df
# )
# three_hop_neighbors_df = inherit_categories_from_seed_df(
    # three_hop_neighbors_df, wikifake_df
# )

# datasets.Dataset.from_pandas(two_hop_neighbors_with_universities_df).save_to_disk(
#     in_data_dir("two_hop_neighbors_with_universities")
# )
# datasets.Dataset.from_pandas(four_hop_neighbors_df).save_to_disk(
#     in_data_dir("four_hop_neighbors_final")
# )
datasets.Dataset.from_pandas(three_hop_neighbors_with_universities_df).save_to_disk(
    in_data_dir("three_hop_neighbors_with_universities_final")
)

Saving the dataset (0/1 shards):   0%|          | 0/169859 [00:00<?, ? examples/s]